In [ ]:
# %% [markdown]
# # ML_NOTEBOOK3 — Dengue Early Warning for Bangladesh
# ## Calibrated forecasting and the resolution-dependence of forecast skill, 2019–2026
#
# Target venue: **7th IEEE ICEEICT 2027**, MIST Dhaka.
# Supersedes `ml_notebook2.py`. Every defect found in the 4 Sep review is fixed here,
# and the additions the plan required (resolution ablation, bootstrap intervals,
# covariate ablation) plus the higher-value ideas (probabilistic baseline, WIS,
# conditional coverage, adaptive conformal, forward test, lead time, Diebold–Mariano)
# are implemented.
#
# ### Required Kaggle inputs
# | Dataset | Files used |
# |---|---|
# | 64-district weekly panel (figshare 10.6084/m9.figshare.33040637) | `Dengue.csv` |
# | Divisional daily bulletin | any CSV with `date, division, Patients, max temp, min temp, rainfall, humidity` |
#
# ### Accelerator
# **Use CPU, not T4 ×2.** Every model here is a gradient-boosted tree on ≤16k rows;
# LightGBM's GPU path is slower than CPU at this size and adds a build dependency.
# The notebook runs end to end in roughly 6–10 minutes on Kaggle's 4-core CPU.

In [ ]:
# =============================================================================
# CELL 0 — Environment, reproducibility, paths
# =============================================================================
import os, sys, glob, json, random, hashlib, logging, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = int(os.environ.get("ICEEICT_SEED", 42))
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)

IS_KAGGLE = os.path.exists("/kaggle")
BASE_INPUT = "/kaggle/input" if IS_KAGGLE else os.path.abspath("./data")
WORKDIR    = "/kaggle/working" if IS_KAGGLE else os.path.abspath(os.environ.get("ICEEICT_OUT", "./artifacts"))
RESULTS    = os.path.join(WORKDIR, "results")
FIGURES    = os.path.join(WORKDIR, "figures")
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger("iceeict")

import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, roc_auc_score,
                             average_precision_score, precision_recall_curve)
from scipy import stats

# ---- Analysis constants. Declared once; used everywhere. --------------------
HORIZONS        = [1, 2, 3, 4]
TEST_YEARS      = [2024, 2025]      # 2026 excluded here; used as a forward test in Cell 12
CAL_WEEKS       = 26                # DEFAULT ONLY - selected empirically in Cell 4b
ALARM_QUANTILE  = 0.80              # per-unit outbreak threshold
ALARM_MIN_CASES = 5
ALARM_TRAIN_MAX = 2023              # threshold derived from <= this year only
ALARM_BASE_MIN  = 2022              # ...and from >= this year. The 2019 rows cover only
                                    # ~30% of that season, so including them shifted the
                                    # per-district quantile away from the modelling window
                                    # and set the alarm base rate off a season no model sees.
NOMINAL         = 0.90
ACI_GAMMA       = 0.02              # DEFAULT ONLY - selected empirically in Cell 4b

# DEFAULTS ONLY. Cell 4b replaces these with values chosen on an inner validation
# split that never sees a test year. Nothing downstream reads them before that.
# deterministic + force_row_wise remove the residual thread-order dependence in
# histogram construction, so a given seed gives the same model on any core count.
LGB_DET = dict(deterministic=True, force_row_wise=True, n_jobs=4)
LGB_REG = dict(n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=31,
               subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
               random_state=SEED, verbose=-1, **LGB_DET)
TWEEDIE_P       = 1.4               # selected in Cell 4b
GROWTH_OBJ_NAME = "L1"              # selected in Cell 4b
ALPHA           = 0.05              # significance level for every test in this notebook
# Single-seed LightGBM estimates move by 2-6 skill points and flip significance
# verdicts between platforms. Every learned model below is a mean over these seeds,
# and table12 reports the seed-to-seed spread of each headline number.
SEEDS           = [int(x) for x in os.environ.get("ICEEICT_SEEDS", "42,7,1").split(",")]
LGB_CLF = dict(n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=31,
               subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
               random_state=SEED, verbose=-1, **LGB_DET)

# Okabe-Ito, ordered so every adjacent pair clears CVD ΔE >= 8 (validated).
PAL = {"blue": "#0072B2", "verm": "#D55E00", "green": "#009E73",
       "orange": "#E69F00", "purple": "#CC79A7", "grey": "#4D4D4D"}

log.info("=" * 78)
log.info("ML_NOTEBOOK3 · ICEEICT 2027 · seed=%d · %s", SEED, "KAGGLE" if IS_KAGGLE else "LOCAL")
log.info("python %s · pandas %s · numpy %s · lightgbm %s",
         sys.version.split()[0], pd.__version__, np.__version__, lgb.__version__)
log.info("results -> %s", RESULTS)
log.info("=" * 78)

_TABLES = {}
def save_table(name, df, caption=""):
    """Persist a results table as CSV and register it in the manifest."""
    p = os.path.join(RESULTS, f"{name}.csv")
    df.to_csv(p, index=False)
    _TABLES[name] = {"rows": len(df), "caption": caption, "path": p}
    log.info("saved %-42s %3d rows", name + ".csv", len(df))
    return df

In [ ]:
# =============================================================================
# CELL 1 — Panel loaders. Both resolutions produce the SAME canonical schema.
# =============================================================================
# canonical columns: unit | block | year | week_start | epi_week | cases | covariates
# `unit`  = the modelling unit (district, or division at coarse resolution)
# `block` = the spatial holdout group (division at both resolutions)

_INPUTS = {}          # every input file this run actually opened, with its hash


def _record_input(path):
    """Hash a resolved input so the manifest pins the data, not just the code.

    _find is the only way an input enters the pipeline, so recording here catches
    every file without relying on anyone remembering to declare it. Two runs whose
    manifests differ here were not run on the same data, whatever else matches.
    """
    try:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        _INPUTS[os.path.relpath(path, BASE_INPUT).replace("\\", "/")] = {
            "sha256": h.hexdigest(), "bytes": os.path.getsize(path)}
    except OSError as e:                       # never let bookkeeping kill a run
        log.warning("could not hash input %s: %s", path, e)
    return path


def _find(*names, root=BASE_INPUT):
    """Locate the first matching file anywhere under root (case-insensitive)."""
    for n in names:
        hits = glob.glob(os.path.join(root, "**", n), recursive=True)
        if hits:
            return _record_input(sorted(hits)[0])
    low = {n.lower() for n in names}
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.lower() in low:
                return _record_input(os.path.join(r, f))
    return None


def load_district_panel():
    p = _find("Dengue.csv", "district_week_panel.csv")
    if p is None:
        raise FileNotFoundError(
            "District panel not found. Attach the 64-district weekly panel "
            "(figshare 10.6084/m9.figshare.33040637) so that Dengue.csv is under /kaggle/input.")
    log.info("district panel  <- %s", p)
    d = pd.read_csv(p)
    d.columns = [c.strip().lower().replace(" ", "_") for c in d.columns]
    d = d.rename(columns={"district": "unit", "division": "block",
                          "new_cases": "cases", "iso_week": "epi_week"})
    d["week_start"] = pd.to_datetime(d["week_start"], errors="coerce")
    d["cases"] = pd.to_numeric(d["cases"], errors="coerce").fillna(0).clip(lower=0)
    # Log-space transforms of the panel's own spillover columns (contemporaneous, legitimate).
    for c in ["nbr_cases", "nbr_cases_lag1", "nbr_cases_lag2",
              "upstream_pressure", "upstream_pressure_lag1", "upstream_pressure_lag2"]:
        if c in d.columns:
            d["log_" + c] = np.log1p(pd.to_numeric(d[c], errors="coerce").clip(lower=0))
    return d.sort_values(["unit", "week_start"]).reset_index(drop=True)


def load_divisional_panel():
    """Aggregate the daily divisional bulletin to weeks ending Saturday."""
    p = _find("divisional_daily_2022_2025.csv", "Dataset (1).csv", "dataset (1).csv")
    if p is None:                                    # last resort: sniff for the schema
        for cand in glob.glob(os.path.join(BASE_INPUT, "**", "*.csv"), recursive=True):
            try:
                cols = {c.strip().lower() for c in pd.read_csv(cand, nrows=2).columns}
            except Exception:
                continue
            if {"date", "division", "patients"} <= cols:
                p = cand; break
    if p is None:
        log.warning("divisional daily file not found — the resolution ablation will be skipped")
        return None
    log.info("divisional panel <- %s", p)
    d = pd.read_csv(p)
    d.columns = [c.strip().lower() for c in d.columns]
    d["date"] = pd.to_datetime(d["date"], format="mixed", errors="coerce")
    d = d.dropna(subset=["date"])
    d["yw"] = d["date"].dt.to_period("W-SAT")
    w = (d.groupby(["division", "yw"])
           .agg(cases=("patients", "sum"), temp_max=("max temp", "mean"),
                temp_min=("min temp", "mean"), rain=("rainfall", "sum"),
                humidity=("humidity", "mean"))
           .reset_index())
    w["week_start"] = w["yw"].dt.start_time
    w["temp_mean"] = (w["temp_max"] + w["temp_min"]) / 2.0

    # The district panel's climate is NASA POWER + CHIRPS. The daily bulletin carries
    # station-style readings instead, which would confound spatial resolution with
    # climate source in the resolution ablation. If the NASA POWER harvest for the
    # eight division centroids is attached, use it and keep the two panels comparable.
    npw = _find("nasa_power_divisions_daily.csv")
    if npw:
        log.info("divisional climate <- NASA POWER (%s)", os.path.basename(npw))
        n = pd.read_csv(npw)
        n["date"] = pd.to_datetime(n["date"], errors="coerce")
        n["yw"] = n["date"].dt.to_period("W-SAT")
        nw = (n.groupby(["division", "yw"])
                .agg(temp_mean=("temp_mean", "mean"), temp_min=("temp_min", "min"),
                     temp_max=("temp_max", "max"), rain=("rain", "sum"),
                     humidity=("humidity", "mean"))
                .reset_index())
        w = (w.drop(columns=["temp_mean", "temp_min", "temp_max", "rain", "humidity"])
               .merge(nw, on=["division", "yw"], how="left"))
        w["climate_source"] = "NASA POWER"
    else:
        log.warning("nasa_power_divisions_daily.csv not attached — the divisional arm will "
                    "use station climate, which confounds resolution with climate source")
        w["climate_source"] = "daily bulletin (station)"
    w["unit"] = w["division"]; w["block"] = w["division"]
    w["year"] = w["week_start"].dt.year
    w["epi_week"] = w["week_start"].dt.isocalendar().week.astype(int)
    w["cases"] = w["cases"].clip(lower=0)
    # Weekly buckets straddle 1 January, so reconcile against the DAILY source instead
    # of the roll-up; otherwise a few days of January land in the previous year.
    w.attrs["daily_annual"] = d.groupby(d["date"].dt.year)["patients"].sum().to_dict()
    return w.drop(columns=["yw", "division"]).sort_values(["unit", "week_start"]).reset_index(drop=True)


PANEL_D = load_district_panel()
PANEL_V = load_divisional_panel()
log.info("district: %s rows, %d units, %s..%s", f"{len(PANEL_D):,}", PANEL_D.unit.nunique(),
         PANEL_D.week_start.min().date(), PANEL_D.week_start.max().date())
if PANEL_V is not None:
    log.info("division: %s rows, %d units, %s..%s", f"{len(PANEL_V):,}", PANEL_V.unit.nunique(),
             PANEL_V.week_start.min().date(), PANEL_V.week_start.max().date())

In [ ]:
# =============================================================================
# CELL 2 — TABLE 1: surveillance reconciliation against published DGHS totals
# =============================================================================
OFFICIAL_DGHS = {2019: 101354, 2022: 62382, 2023: 321179, 2024: 101214}
# Reconciliation tolerance. The district panel is a curated deposit reconciled by its
# author against the DGHS bulletins; the divisional file is a Kaggle mirror of the same
# bulletins and is known to run ~1.9% light in 2022, so it gets a looser bound.
RECON_TOL_PCT = {"district": 1.0, "divisional": 2.5}

def reconcile(panel, tag):
    rows = []
    annual = panel.attrs.get("daily_annual") or panel.groupby("year")["cases"].sum().to_dict()
    for y, obs in sorted(annual.items()):
        off = OFFICIAL_DGHS.get(int(y))
        d = (obs - off) / off * 100 if off else np.nan
        # 2019 is a partial season in the district panel (rows begin 26 Aug), so a
        # full-year match is not expected and the deposit's README overstates it.
        first, last = panel.week_start.min(), panel.week_start.max()
        partial = (int(y) == 2019) or (int(y) >= 2025)                   or first > pd.Timestamp(f"{int(y)}-01-08")                   or last < pd.Timestamp(f"{int(y)}-12-24")
        rows.append({"panel": tag, "year": int(y), "observed": int(obs),
                     "official_dghs": off if off else np.nan,
                     "delta_pct": round(d, 2) if off else np.nan,
                     "abs_delta_pct": abs(d) if off else np.nan,
                     "tolerance_pct": RECON_TOL_PCT.get(tag, 1.0),
                     "status": "partial season" if partial else
                               ("reconciled" if abs(d) < RECON_TOL_PCT.get(tag, 1.0)
                                else "DISCREPANCY")})
    return pd.DataFrame(rows)

recon = reconcile(PANEL_D, "district")
if PANEL_V is not None:
    recon = pd.concat([recon, reconcile(PANEL_V, "divisional")], ignore_index=True)
save_table("table1_reconciliation", recon, "DGHS surveillance reconciliation")
print(recon.to_string(index=False))

# ---- Data-quality audit. Runs on every execution so the paper's Data section
# ---- can be written from the run output rather than from assertion.
def data_quality_audit(d):
    r = []
    def chk(name, value, status, note=""):
        r.append({"check": name, "value": value, "status": status, "note": note})

    grid = d["unit"].nunique() * d["week_start"].nunique()
    chk("panel completeness", f"{len(d):,} of {grid:,} cells",
        "PASS" if len(d) == grid else "FLAG", "64 units x 254 ISO weeks, Monday-start")
    chk("duplicate unit-weeks", int(d.duplicated(["unit", "week_start"]).sum()), "PASS", "")
    chk("negative or fractional cases",
        int((d.cases < 0).sum() + (d.cases % 1 != 0).sum()), "PASS", "")

    # Zeros that are probably missing reports rather than true zeros.
    gap = (d.cases == 0) & (d.get("currently_admitted", pd.Series(0, index=d.index)) > 0)
    chk("zero-case weeks", f"{100*(d.cases == 0).mean():.1f}%", "INFO", "")
    chk("zeros that look like missing reports", f"{int(gap.sum()):,} rows "
        f"({100*gap.sum()/max((d.cases == 0).sum(),1):.1f}% of zeros)", "FLAG",
        "cases=0 while patients are still admitted; treat zero-inflation claims with care")

    if "new_deaths" in d:
        chk("weeks with deaths but no new cases", int((d.new_deaths > d.cases).sum()), "FLAG",
            "reporting lag: deaths of patients admitted earlier")
    if "cumulative_cases" in d:
        dec = sum(int((g.sort_values("week_start").cumulative_cases.diff().dropna() < 0).any())
                  for _, g in d.groupby(["unit", "year"]))
        chk("seasons where cumulative_cases decreases", f"{dec} of {d.groupby(['unit','year']).ngroups}",
            "FLAG", "one per season at the ISO-year boundary; column is excluded from features")

    for c, lo, hi in [("temp_mean", 5, 42), ("humidity", 10, 100), ("rain", 0, 1500),
                      ("ndvi", -1, 1), ("ndwi", -1, 1)]:
        if c in d:
            chk(f"range check: {c}", f"[{d[c].min():.2f}, {d[c].max():.2f}]",
                "PASS" if ((d[c] >= lo) & (d[c] <= hi)).all() else "FLAG", f"expected [{lo}, {hi}]")
    if {"temp_min", "temp_mean", "temp_max"} <= set(d.columns):
        ok = ((d.temp_min <= d.temp_mean) & (d.temp_mean <= d.temp_max)).all()
        chk("temp_min <= mean <= max", "holds" if ok else "violated", "PASS" if ok else "FLAG", "")

    if "gtrends_dengue" in d:
        per_wk = d.groupby("week_start").gtrends_dengue.nunique().median()
        chk("gtrends_dengue distinct values per week", f"{per_wk:.0f} across {d.unit.nunique()} units",
            "FLAG", "effectively a national seasonal clock; describe it as such, not as district search behaviour")

    nz = d.loc[d.cases > 0, "cases"].astype(int).astype(str).str[0].value_counts(normalize=True)
    benford = pd.Series({str(i): np.log10(1 + 1 / i) for i in range(1, 10)})
    dev = float((nz - benford).abs().max())
    chk("Benford deviation, nonzero counts", f"{dev:.3f}",
        "PASS" if dev < 0.05 else "FLAG", "large deviations would suggest synthetic counts")
    return pd.DataFrame(r)

dq = save_table("table1b_data_quality", data_quality_audit(PANEL_D),
                "Data-quality audit of the district panel")
print(dq.to_string(index=False))


# ---- External cross-checks. The internal audit above cannot detect a dataset that
# ---- is internally consistent but wrong. These two compare the panel against
# ---- sources it was NOT derived from.
def _isokey(s):
    ic = pd.to_datetime(s).dt.isocalendar()
    return list(zip(ic.year.astype(int), ic.week.astype(int)))


def external_cross_checks(panel):
    """Panel cases vs an independently compiled bulletin; panel climate vs NASA POWER.

    Everything is aligned on the ISO (year, week) key. The district panel uses
    Monday-start ISO weeks while the divisional roll-up uses weeks ending Saturday,
    so merging on a date column silently returns zero rows.
    """
    rows = []
    panel = panel.copy()
    panel["k"] = _isokey(panel["week_start"])

    # --- 1. cases vs the independent daily divisional bulletin
    bp = _find("divisional_daily_2022_2025.csv", "Dataset (1).csv", "dataset (1).csv")
    if bp:
        b = pd.read_csv(bp)
        b.columns = [c.strip().lower() for c in b.columns]
        b["date"] = pd.to_datetime(b["date"], format="mixed", errors="coerce")
        b = b.dropna(subset=["date"]); b["k"] = _isokey(b["date"])
        bw = b.groupby(["division", "k"])["patients"].sum().rename("bulletin").reset_index()
        pw = panel.groupby(["block", "k"])["cases"].sum().rename("panel").reset_index()                   .rename(columns={"block": "division"})
        m = pw.merge(bw, on=["division", "k"])
        m = m[m["k"].map(lambda t: 2022 <= t[0] <= 2025)]
        if len(m):
            rows += [
                {"check": "cases vs independent bulletin", "metric": "Pearson r",
                 "value": round(float(m.panel.corr(m.bulletin)), 4),
                 "n": len(m), "verdict": "PASS" if m.panel.corr(m.bulletin) > 0.99 else "FLAG"},
                {"check": "cases vs independent bulletin", "metric": "exact-match rate",
                 "value": round(float((m.panel == m.bulletin).mean()), 4), "n": len(m),
                 "verdict": "INFO"},
                {"check": "cases vs independent bulletin", "metric": "total difference %",
                 "value": round(float(100 * (m.panel.sum() - m.bulletin.sum()) / m.bulletin.sum()), 3),
                 "n": len(m), "verdict": "PASS" if abs(m.panel.sum() - m.bulletin.sum()) /
                 m.bulletin.sum() < 0.02 else "FLAG"}]
    else:
        rows.append({"check": "cases vs independent bulletin", "metric": "not run",
                     "value": np.nan, "n": 0, "verdict": "SKIPPED — divisional file not attached"})

    # --- 2. climate vs an independent NASA POWER harvest
    np_path = _find("nasa_power_divisions_daily.csv")
    if np_path:
        n = pd.read_csv(np_path)
        n["date"] = pd.to_datetime(n["date"], errors="coerce"); n["k"] = _isokey(n["date"])
        nw = (n.groupby(["division", "k"])
                .agg(t_nasa=("temp_mean", "mean"), h_nasa=("humidity", "mean"),
                     r_nasa=("rain", "sum")).reset_index())
        dc = (panel.groupby(["block", "k"])
                .agg(t_panel=("temp_mean", "mean"), h_panel=("humidity", "mean"),
                     r_panel=("rain", "mean")).reset_index().rename(columns={"block": "division"}))
        c = dc.merge(nw, on=["division", "k"])
        for a, b_, lab, thr in [("t_panel", "t_nasa", "temp_mean", 0.97),
                                ("h_panel", "h_nasa", "humidity", 0.95),
                                ("r_panel", "r_nasa", "rain", 0.95)]:
            r_ = float(c[a].corr(c[b_])) if len(c) else np.nan
            rows.append({"check": f"climate vs NASA POWER: {lab}", "metric": "Pearson r",
                         "value": round(r_, 4), "n": len(c),
                         "verdict": "PASS" if r_ > thr else
                         "FLAG — different product (panel rain is CHIRPS, not POWER)"})
            rows.append({"check": f"climate vs NASA POWER: {lab}", "metric": "mean bias (panel - NASA)",
                         "value": round(float((c[a] - c[b_]).mean()), 3), "n": len(c), "verdict": "INFO"})
    else:
        rows.append({"check": "climate vs NASA POWER", "metric": "not run", "value": np.nan,
                     "n": 0, "verdict": "SKIPPED — nasa_power_divisions_daily.csv not attached"})
    return pd.DataFrame(rows)


xc = save_table("table1c_external_cross_checks", external_cross_checks(PANEL_D),
                "Panel validated against sources it was not derived from")
print(xc.to_string(index=False))

zero_share = (PANEL_D.cases == 0).mean()
log.info("district zero-case share: %.1f%%  |  cumulative cases: %s",
         100 * zero_share, f"{int(PANEL_D.cases.sum()):,}")

In [ ]:
# =============================================================================
# CELL 3 — Feature engineering. Season-blocked so lags never bridge a data gap.
# =============================================================================
CLIM_BASE = ["rain", "temp_mean", "temp_min", "temp_max", "humidity"]


def engineer(panel):
    """Add lags, growth deltas, rolling stats, harmonics and targets.

    Lags are computed within (unit, block) where `block` marks a contiguous
    stretch of weeks. The district panel jumps 2019 -> 2022, so a plain shift
    would make the first week of 2022 look back at the last week of 2019.
    """
    df = panel.copy()
    df["gap_block"] = np.where(df["year"] <= 2019, "pre", "post")
    out = []
    for _, g in df.groupby(["unit", "gap_block"], sort=False):
        g = g.sort_values("week_start").copy()
        g["cases_lag0"] = g["cases"]
        g["log_cases_lag0"] = np.log1p(g["cases"])
        for L in [1, 2, 3, 4, 8]:
            g[f"cases_lag{L}"] = g["cases"].shift(L)
            g[f"log_cases_lag{L}"] = np.log1p(g[f"cases_lag{L}"])
        g["growth_delta_1w"] = g["log_cases_lag0"] - g["log_cases_lag1"]
        g["growth_delta_2w"] = g["log_cases_lag0"] - g["log_cases_lag2"]
        g["growth_delta_4w"] = g["log_cases_lag0"] - g["log_cases_lag4"]
        g["rolling_mean_log_4w"] = g["log_cases_lag0"].rolling(4, min_periods=2).mean()
        g["rolling_std_log_4w"]  = g["log_cases_lag0"].rolling(4, min_periods=2).std().fillna(0)
        g["rolling_mean_log_8w"] = g["log_cases_lag0"].rolling(8, min_periods=4).mean()
        g["dev_from_mean_4w"]    = g["log_cases_lag0"] - g["rolling_mean_log_4w"]
        # Climate lags recomputed here rather than taken from the deposit, so the
        # definition is byte-identical at both resolutions.
        for c in CLIM_BASE:
            if c in g.columns:
                for L in (2, 3, 4):
                    g[f"{c}_ownlag{L}"] = g[c].shift(L)
        for h in HORIZONS:
            g[f"target_lead_{h}w"]   = g["cases"].shift(-h)
            g[f"target_growth_{h}w"] = np.log((g["cases"].shift(-h) + 1) / (g["cases"] + 1))
        out.append(g)
    df = pd.concat(out).reset_index(drop=True)
    df["sin_epi_week"] = np.sin(2 * np.pi * df["epi_week"] / 52.1775)
    df["cos_epi_week"] = np.cos(2 * np.pi * df["epi_week"] / 52.1775)

    return df


def add_alarm_labels(M, quantile=None, floor=None):
    """Attach the outbreak threshold and alarm labels to a MODELLING frame.

    Deliberately not done inside engineer(). The threshold has to be computed over
    exactly the rows that are modelled: engineer() still holds the 2019 partial
    season and the early-2022 weeks that later drop out for want of an 8-week lag,
    and both pull the per-district quantile away from the modelling window. Doing it
    here also means the Cell 9b sweep and the reported matrix share one definition,
    which a gate then verifies.
    """
    M = M.copy()
    q = quantile if quantile is not None else ALARM_QUANTILE
    fl = ALARM_MIN_CASES if floor is None else floor
    base = M[(M.year >= ALARM_BASE_MIN) & (M.year <= ALARM_TRAIN_MAX)]
    thr_map = base.groupby("unit")["cases"].quantile(q)
    # How often does the floor override the quantile? At low quantiles many quiet
    # districts sit below it, so a "q = 0.70" threshold is partly a floor of 5 and
    # the sweep row must say so rather than be read as a pure quantile.
    M.attrs["n_floored"] = int((thr_map < fl).sum())
    M.attrs["n_units"] = int(thr_map.shape[0])
    thr = np.maximum(M["unit"].map(thr_map).fillna(fl), fl)
    M["alarm_threshold"] = thr
    for h in HORIZONS:
        lead = M[f"target_lead_{h}w"]
        M[f"target_alarm_{h}w"] = (lead >= thr).astype(float).where(lead.notna())
    return M


# ---- Feature groups. An ALLOW-LIST: nothing enters the model unless named here.
G_AR = ["cases_lag0", "log_cases_lag0",
        "cases_lag1", "cases_lag2", "cases_lag3", "cases_lag4", "cases_lag8",
        "log_cases_lag1", "log_cases_lag2", "log_cases_lag3", "log_cases_lag4", "log_cases_lag8",
        "growth_delta_1w", "growth_delta_2w", "growth_delta_4w",
        "rolling_mean_log_4w", "rolling_std_log_4w", "rolling_mean_log_8w", "dev_from_mean_4w"]
G_SEASON   = ["sin_epi_week", "cos_epi_week", "epi_week"]
# Shared climate: base variables plus lags recomputed identically on both panels.
# Source is NASA POWER (+ CHIRPS precipitation in the district deposit) on both sides.
G_CLIMATE  = (CLIM_BASE
              + [f"{c}_ownlag{L}" for c in CLIM_BASE for L in (2, 3, 4)])
# District-only extras shipped by the deposit (its own lag definitions, plus CHIRPS
# precipitation power). Excluded from the shared set so the ablation stays like-for-like.
G_CLIMATE_EXTRA = ["precip_power", "rain_lag2", "rain_lag3", "rain_lag4",
                   "temp_mean_lag2", "temp_mean_lag3", "temp_mean_lag4",
                   "humidity_lag2", "humidity_lag3", "humidity_lag4"]
G_SAT      = ["ndvi", "ndwi", "lst", "viirs", "ndvi_filled", "viirs_change"]
G_STATIC   = ["pop_density", "river_density", "road_density", "population"]
G_TRENDS   = ["gtrends_dengue", "gtrends_dengue_nat", "gtrends_dengue_bn_nat", "gtrends_fever_bn_nat"]
G_MOBILITY = ["log_nbr_cases", "log_nbr_cases_lag1", "log_nbr_cases_lag2",
              "log_upstream_pressure", "log_upstream_pressure_lag1", "log_upstream_pressure_lag2",
              "festival_window", "festival_influx"]
G_SURV     = ["new_deaths", "currently_admitted", "incidence"]

# Never features. `ignition*` and `eligible*` are the deposit's own forward-looking
# labels; the cumulative counters are monotone season identifiers that will not transfer.
FORBIDDEN = {"ignition", "ignition_loose", "eligible", "eligible_loose", "naive",
             "cumulative_cases", "cumulative_deaths", "alarm_threshold",
             "cases", "unit", "block", "year", "week_start", "gap_block", "district_id"}

SHARED_GROUPS = [G_AR, G_SEASON, G_CLIMATE]                       # available at BOTH resolutions
FULL_GROUPS   = SHARED_GROUPS + [G_CLIMATE_EXTRA, G_SAT, G_STATIC, G_TRENDS, G_MOBILITY, G_SURV]

def select_features(df, groups):
    allow, seen = [], set()
    for g in groups:
        for c in g:
            if c in df.columns and c not in FORBIDDEN and c not in seen:
                if pd.api.types.is_numeric_dtype(df[c]):
                    allow.append(c); seen.add(c)
    return allow


def burden_groups(M, n_groups=3):
    """Assign each unit to a burden tertile using TRAINING years only.

    Used for group-conditional (Mondrian) conformal calibration. Marginal coverage
    of 0.86 turned out to be a mixture of ~0.90 in low-burden districts and ~0.83 in
    high-burden ones, so one shared correction is the wrong object: the score
    distribution genuinely differs by burden.
    """
    b = (M[(M.year >= ALARM_BASE_MIN) & (M.year <= ALARM_TRAIN_MAX)]
         .groupby("unit")["cases"].mean())
    if b.nunique() < n_groups:
        return {u: 0 for u in M.unit.unique()}
    lab = pd.qcut(b, n_groups, labels=False, duplicates="drop")
    return {u: int(v) for u, v in lab.items()}


def prepare(panel, groups, min_year=2022):
    df = engineer(panel)
    F = select_features(df, groups)
    M = df[(df.year >= min_year) & df[f"target_growth_1w"].notna() & df["cases_lag8"].notna()].copy()
    M[F] = M[F].fillna(0.0)
    M = add_alarm_labels(M.reset_index(drop=True))
    return M, F


MD, FD_FULL = prepare(PANEL_D, FULL_GROUPS)          # district, all covariates
_,  FD_SHR  = prepare(PANEL_D, SHARED_GROUPS)        # district, shared covariates only
MV, FV_SHR  = (prepare(PANEL_V, SHARED_GROUPS) if PANEL_V is not None else (None, []))

log.info("district model frame: %s rows | %d full features | %d shared features",
         f"{len(MD):,}", len(FD_FULL), len(FD_SHR))
if MV is not None:
    log.info("division model frame: %s rows | %d shared features", f"{len(MV):,}", len(FV_SHR))

In [ ]:
# =============================================================================
# CELL 4 — Integrity gate. Every defect from the 4 Sep review trips one of these.
# =============================================================================
def gate_data(M, F, recon_df, tag):
    assert not (FORBIDDEN & set(F)), f"[{tag}] forbidden column in feature set"
    assert not any(c.startswith("target_") for c in F), f"[{tag}] target leaked into features"
    bad = recon_df[(recon_df.panel == tag) & (recon_df.status == "DISCREPANCY")]
    assert bad.empty, f"[{tag}] reconciliation failed:\n{bad}"
    # no lag may reach across the 2020-21 gap
    early = M[(M.year == 2022) & (M.epi_week <= 8)]
    if len(early):
        assert early["cases_lag8"].notna().sum() == 0 or "pre" not in set(M.gap_block), \
            f"[{tag}] lag features bridge the 2020-21 gap"
    assert M.groupby(["unit", "week_start"]).size().max() == 1, f"[{tag}] duplicate unit-week rows"
    log.info("[gate] %-10s OK — %d features, %s rows", tag, len(F), f"{len(M):,}")

gate_data(MD, FD_FULL, recon, "district")
if MV is not None:
    gate_data(MV, FV_SHR, recon, "divisional")
    missing = set(FD_SHR) ^ set(FV_SHR)
    assert not missing, ("resolution ablation is not like-for-like; feature sets differ by "
                         f"{sorted(missing)}")
    log.info("[gate] shared feature set identical at both resolutions (%d features)", len(FD_SHR))

In [ ]:
# =============================================================================
# CELL 4b — Empirical model selection on an INNER split (never sees a test year)
# =============================================================================
# Selection train = 2019 tail + 2022, selection validation = 2023.
# 2024 and 2025 are untouched here, so nothing downstream is tuned on its own test set.
from sklearn.model_selection import ParameterGrid

MSEL, FSEL = prepare(PANEL_D, FULL_GROUPS, min_year=2019)
SEL_TRAIN, SEL_VALID = 2022, 2023
_tr = MSEL[(MSEL.year <= SEL_TRAIN) & MSEL["target_lead_2w"].notna()]
_va = MSEL[(MSEL.year == SEL_VALID) & MSEL["target_lead_2w"].notna()]
log.info("selection split: train %s rows (<=%d), validate %s rows (%d)",
         f"{len(_tr):,}", SEL_TRAIN, f"{len(_va):,}", SEL_VALID)

def _score(params, target, extra=None, h=2):
    """Mean and standard error of inner-validation MAE across SEEDS.

    One seed is not enough. Candidate scores here sit within a few MAE points of
    each other while seed-to-seed noise is of the same order, so a single-seed
    argmin selects on noise and flips between machines.
    """
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    col = tl if target == "level" else tg
    vals = []
    for sd in SEEDS:
        m = lgb.LGBMRegressor(**{**params, "random_state": sd, "verbose": -1, **LGB_DET},
                              **(extra or {})).fit(_tr[FSEL], _tr[col])
        pred = m.predict(_va[FSEL])
        if target == "growth":
            pred = (_va["cases_lag0"].values + 1.0) * np.exp(pred) - 1.0
        vals.append(mean_absolute_error(_va[tl], np.clip(pred, 0, None)))
    v = np.asarray(vals, float)
    se = float(v.std(ddof=1) / np.sqrt(len(v))) if len(v) > 1 else 0.0
    return float(v.mean()), se


def _pick_1se(scored, simpler_key):
    """One-standard-error rule: among candidates within 1 s.e. of the best mean,
    take the simplest. Near-ties then resolve deterministically, not by noise."""
    best = min(scored, key=lambda r: r["mean"])
    tied = [r for r in scored if r["mean"] <= best["mean"] + best["se"]]
    return min(tied, key=lambda r: simpler_key(r["cand"]))

# max_depth=5 caps the tree at 2**5 = 32 leaves, so num_leaves of 31 and 63 fit
# byte-identical models. Candidates above the cap are dropped, not scored twice.
GRID = [g for g in ParameterGrid({"num_leaves": [7, 15, 31], "min_child_samples": [10, 20, 40],
                                  "learning_rate": [0.05], "n_estimators": [300],
                                  "max_depth": [5], "subsample": [0.8], "colsample_bytree": [0.8]})
        if g["num_leaves"] <= 2 ** g["max_depth"] - 1]
sel_rows = []

# (a) tree hyperparameters, chosen once on the growth target and reused for both,
#     so the two paradigms cannot differ merely through their tuning budget.
_scored = []
for g in GRID:
    mu, se = _score(g, "growth")
    _scored.append({"cand": g, "mean": mu, "se": se})
# "simplest" = fewest leaves, then most regularised
best = _pick_1se(_scored, lambda g: (g["num_leaves"], -g["min_child_samples"]))["cand"]
for r in _scored:
    sel_rows.append({"decision": "tree hyperparameters", "candidate": str(
        {k: r["cand"][k] for k in ("num_leaves", "min_child_samples")}),
        "inner_valid_MAE": round(r["mean"], 4), "se": round(r["se"], 4),
        "selected": r["cand"] == best})
LGB_REG = {**best, "random_state": SEED, "verbose": -1, "n_jobs": -1}
log.info("selected hyperparameters: %s", {k: best[k] for k in ("num_leaves", "min_child_samples")})

# (b) Tweedie variance power for the level target
_tw = []
for q in [1.1, 1.3, 1.5, 1.7, 1.9]:
    mu, se = _score(best, "level", dict(objective="tweedie", tweedie_variance_power=q))
    _tw.append({"cand": q, "mean": mu, "se": se})
TWEEDIE_P = _pick_1se(_tw, lambda q: q)["cand"]
sel_rows += [{"decision": "tweedie_variance_power", "candidate": str(r["cand"]),
              "inner_valid_MAE": round(r["mean"], 4), "se": round(r["se"], 4),
              "selected": r["cand"] == TWEEDIE_P} for r in _tw]
log.info("selected tweedie_variance_power: %.1f", TWEEDIE_P)

# (c) objective for the anchored-growth target
_gr = []
for k, o in [("L1", dict(objective="regression_l1")), ("L2", dict(objective="regression"))]:
    mu, se = _score(best, "growth", o)
    _gr.append({"cand": k, "mean": mu, "se": se})
GROWTH_OBJ_NAME = min(_gr, key=lambda r: r["mean"])["cand"]
sel_rows += [{"decision": "growth objective", "candidate": r["cand"],
              "inner_valid_MAE": round(r["mean"], 4), "se": round(r["se"], 4),
              "selected": r["cand"] == GROWTH_OBJ_NAME} for r in _gr]
log.info("selected growth objective: %s", GROWTH_OBJ_NAME)

# (d) conformal calibration window, scored by |coverage - nominal| on the inner split
def _cal_score(cw, h=2, seed=None):
    # _tr and _va are masked on target_lead_2w, so at h=4 they still carry rows whose
    # 4-week targets are NaN (season blocking ends each block early). Those NaNs reached
    # np.quantile, made the conformal quantile NaN, and every interval comparison then
    # evaluated False - coverage 0.0, score |0 - 0.9| = 0.9 for BOTH candidate windows,
    # so the h=4 window was being picked by min() breaking a tie between two failures.
    tg, tl = f"target_growth_{h}w", f"target_lead_{h}w"
    tra = _tr[_tr[tg].notna() & _tr[tl].notna()].sort_values("week_start")
    val = _va[_va[tg].notna() & _va[tl].notna()]
    if len(val) < 100:
        return np.inf
    wk = np.sort(tra["week_start"].unique())
    if len(wk) <= cw + 4:
        return np.inf
    idx = np.unique(np.linspace(0, len(wk) - 1, cw).round().astype(int))
    is_cal = tra["week_start"].isin(set(wk[idx]))
    tr2, cal = tra[~is_cal], tra[is_cal]
    if len(tr2) < 300 or len(cal) < 100:
        return np.inf
    q = {a: lgb.LGBMRegressor(**{**LGB_REG, "random_state": seed if seed is not None else SEED},
                              objective="quantile", alpha=a).fit(tr2[FSEL], tr2[tg])
         for a in (0.05, 0.95)}
    cl, ch = q[0.05].predict(cal[FSEL]), q[0.95].predict(cal[FSEL])
    _wf = max(float(np.quantile(ch - cl, 0.05)), 1e-3)
    cw_ = np.maximum(ch - cl, _wf)
    sc = np.maximum(cl - cal[tg].values, cal[tg].values - ch) / cw_
    n = len(sc); qh = np.quantile(sc, min(1.0, np.ceil((n + 1) * NOMINAL) / n))
    if not np.isfinite(qh):
        return np.inf                      # unusable, and must not look like a good score
    anc = val["cases_lag0"].values + 1.0
    vl, vh = q[0.05].predict(val[FSEL]), q[0.95].predict(val[FSEL])
    vw = np.maximum(vh - vl, _wf)
    lo = anc * np.exp(vl - qh * vw) - 1
    hi = anc * np.exp(vh + qh * vw) - 1
    y = val[tl].values
    return abs(((y >= np.clip(lo, 0, None)) & (y <= hi)).mean() - NOMINAL)

# Selected PER HORIZON: the score is |coverage - nominal|, and a 1-week-ahead
# residual distribution is not the same object as a 4-week-ahead one.
CAL_WEEKS_BY_H = {}
for _h in [1, 2, 4]:
    # The conformal quantile is estimated from calibration WEEKS, not rows: the 64
    # districts in a week are far from independent. An 8-week window gives ~8 effective
    # observations for a 90% quantile, which is why it produced a no-op correction.
    # Candidates are floored at 26 weeks and capped so the training half stays larger.
    _sc = {cw: float(np.mean([_cal_score(cw, h=_h, seed=sd) for sd in SEEDS]))
           for cw in [26, 39, 52]}
    _sc = {cw: v for cw, v in _sc.items() if np.isfinite(v)}   # drop infeasible windows
    assert _sc, f"no feasible calibration window at h={_h}"
    # A selection between two identical scores is not a selection. This fired at h=4,
    # where both candidates scored exactly 0.9 because neither produced usable intervals.
    if len(_sc) > 1 and len(set(np.round(list(_sc.values()), 6))) == 1:
        raise AssertionError(
            f"every calibration window scores identically at h={_h} ({_sc}) - the "
            "selection is arbitrary and the resulting intervals cannot be reported")
    CAL_WEEKS_BY_H[_h] = min(_sc, key=_sc.get)
    sel_rows += [{"decision": f"conformal calibration weeks (h={_h})", "candidate": str(cw),
                  "inner_valid_MAE": np.nan, "coverage_deviation": round(v, 4),
                  "se": np.nan,
                  "selected": cw == CAL_WEEKS_BY_H[_h]} for cw, v in _sc.items()]
    log.info("  h=%d calibration windows scored: %s", _h, sorted(_sc))
CAL_WEEKS = CAL_WEEKS_BY_H[2]
log.info("selected conformal calibration window per horizon: %s", CAL_WEEKS_BY_H)

# (e) adaptive-conformal step size, on the same inner split. Without this gamma is
# just a number someone picked; with it, the drift correction is tuned to how fast
# this system actually drifts.
def _gamma_score(gm, h=2):
    tg, tl = f"target_growth_{h}w", f"target_lead_{h}w"
    tra = _tr.sort_values("week_start")
    wk = np.sort(tra["week_start"].unique()); cw = CAL_WEEKS_BY_H.get(h, 26)
    if len(wk) <= cw + 4:
        return np.inf
    idx = np.unique(np.linspace(0, len(wk) - 1, cw).round().astype(int))
    isc = tra["week_start"].isin(set(wk[idx]))
    tr2, cal = tra[~isc], tra[isc]
    q = {a: lgb.LGBMRegressor(**LGB_REG, objective="quantile", alpha=a).fit(tr2[FSEL], tr2[tg])
         for a in (0.05, 0.95)}
    cl, ch = q[0.05].predict(cal[FSEL]), q[0.95].predict(cal[FSEL])
    cwd = np.maximum(ch - cl, 1e-6)
    sc = np.maximum(cl - cal[tg].values, cal[tg].values - ch) / cwd
    va = _va.sort_values("week_start")
    vl, vh = q[0.05].predict(va[FSEL]), q[0.95].predict(va[FSEL])
    vw = np.maximum(vh - vl, 1e-6); anc = va["cases_lag0"].values + 1.0
    y = va[tl].values; weeks = va["week_start"].values
    alpha_t, cov = 1 - NOMINAL, np.zeros(len(va), bool)
    for wkk in pd.unique(weeks):
        sel = weeks == wkk
        o = np.quantile(sc, float(np.clip(1 - alpha_t, 0.01, 0.999))) * vw[sel]
        c = (y[sel] >= anc[sel] * np.exp(vl[sel] - o) - 1) & (y[sel] <= anc[sel] * np.exp(vh[sel] + o) - 1)
        cov[sel] = c
        alpha_t += gm * ((1 - NOMINAL) - (1 - c.mean()))
    return abs(cov.mean() - NOMINAL)

_gs = {g: _gamma_score(g) for g in [0.005, 0.01, 0.02, 0.05, 0.10]}
ACI_GAMMA = min(_gs, key=_gs.get)
sel_rows += [{"decision": "adaptive conformal step size (gamma)", "candidate": str(g),
              "inner_valid_MAE": round(v, 4), "se": np.nan, "selected": g == ACI_GAMMA}
             for g, v in _gs.items()]
log.info("selected adaptive-conformal gamma: %s", ACI_GAMMA)

save_table("table1d_model_selection", pd.DataFrame(sel_rows),
           f"Every tuned constant, chosen on train<={SEL_TRAIN} / validate {SEL_VALID}; "
           f"metric is inner-validation MAE except the calibration window (|coverage-nominal|)")
del MSEL, FSEL, _tr, _va

In [ ]:
# =============================================================================
# CELL 5 — Metrics, rolling-origin harness, and the model zoo
# =============================================================================
class _EnsQ:
    """Mean prediction over a list of fitted models, with a .predict interface."""
    def __init__(self, models): self.models = models
    def predict(self, X): return np.mean([m.predict(X) for m in self.models], axis=0)


from statsmodels.tsa.arima.model import ARIMA
import statsmodels.api as sm
warnings.filterwarnings("ignore", module="statsmodels")

ARIMA_ORDER = (2, 1, 2)   # Naher et al. (2022) selected ARIMA(2,1,2) for Bangladeshi
                          # dengue by AIC/BIC - but on NATIONAL MONTHLY counts, a smooth
                          # series. Applied here at district-week resolution, where 47% of
                          # observations are zero, it is outside the regime it was chosen
                          # for. Report it as "the published benchmark does not transfer to
                          # the operational resolution", never as "our model beats ARIMA":
                          # the order was not re-selected for this data and doing so would
                          # be the fair comparison.
_ARIMA_CACHE = {}


def arima_baseline(M, h, test_years=TEST_YEARS, tag=""):
    """Per-unit ARIMA on log1p counts, rolled through the test year without refitting.

    At each test-week origin the model forecasts h steps ahead, then the observed
    value is appended (refit=False, so the state updates but the parameters do not).
    That mirrors how the tree models are used: parameters estimated once per fold on
    strictly earlier data, then applied forward.
    """
    key = (tag, h, tuple(test_years))
    if key in _ARIMA_CACHE:
        return _ARIMA_CACHE[key]
    rows = []
    for ty in test_years:
        for u, g in M[M.year <= ty].groupby("unit"):
            g = g.sort_values("week_start")
            tr = g[g.year < ty]
            te = g[(g.year == ty) & g[f"target_lead_{h}w"].notna()]
            if len(tr) < 30 or not len(te):
                continue
            y_tr = np.log1p(tr["cases"].values.astype(float))
            try:
                res = ARIMA(y_tr, order=ARIMA_ORDER).fit()
            except Exception:
                continue
            y_te = np.log1p(te["cases"].values.astype(float))
            for i, (_, r) in enumerate(te.iterrows()):
                try:
                    f = float(res.forecast(h)[-1])
                except Exception:
                    f = y_te[i]
                rows.append({"unit": u, "week_start": r["week_start"],
                             "arima": max(0.0, float(np.expm1(f)))})
                try:
                    res = res.append([y_te[i]], refit=False)
                except Exception:
                    break
    out = pd.DataFrame(rows)
    _ARIMA_CACHE[key] = out
    return out


def metrics(y, p):
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 0, None)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    return dict(MAE=mean_absolute_error(y, p),
                RMSE=float(np.sqrt(np.mean((y - p) ** 2))),
                wMAPE=float(100 * np.abs(y - p).sum() / y.sum()) if y.sum() > 0 else np.nan)

# Each target gets a sweep of plausible objectives, so neither side of the
# comparison can be accused of running on a misspecified likelihood.
LEVEL_OBJECTIVES = {
    "L2":      dict(objective="regression"),
    "Poisson": dict(objective="poisson"),
    "Tweedie": dict(objective="tweedie", tweedie_variance_power=TWEEDIE_P),
}
GROWTH_OBJECTIVES = {
    "L2": dict(objective="regression"),
    "L1": dict(objective="regression_l1"),   # median of the log-ratio
}

def rolling_origin(M, F, h, test_years=TEST_YEARS, arima_tag=None, with_extras=False):
    """Expanding-window rolling origin. Returns one long frame of aligned predictions."""
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    frames = []
    for ty in test_years:
        tr = M[(M.year < ty) & M[tl].notna()]
        te = M[(M.year == ty) & M[tl].notna()]
        if len(te) < 20 or len(tr) < 100:
            continue
        out = te[["unit", "block", "year", "epi_week", "week_start", "cases_lag0", tl]].copy()
        out = out.rename(columns={tl: "y"})
        out["fold"] = ty
        out["persistence"] = te["cases_lag0"].values

        def _ens(col, obj, transform=None):
            """Mean prediction over SEEDS. Also returns the per-seed predictions so
            Cell 12c can report how much a single-seed number would have moved."""
            ps = []
            for sd in SEEDS:
                m = lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd}, **obj).fit(tr[F], tr[col])
                q = m.predict(te[F])
                ps.append(transform(q) if transform else q)
            return np.clip(np.mean(ps, axis=0), 0, None), [np.clip(x, 0, None) for x in ps]

        anchor = te["cases_lag0"].values + 1.0
        for name, obj in LEVEL_OBJECTIVES.items():
            mean_p, per_seed = _ens(tl, obj)
            out[f"level_{name}"] = mean_p
            for sd, ps in zip(SEEDS, per_seed):
                out[f"__seed{sd}__level_{name}"] = ps

        for name, obj in GROWTH_OBJECTIVES.items():
            mean_p, per_seed = _ens(tg, obj, lambda g: anchor * np.exp(g) - 1.0)
            out[f"anchored_{name}"] = mean_p
            for sd, ps in zip(SEEDS, per_seed):
                out[f"__seed{sd}__anchored_{name}"] = ps
        out["anchored"] = out[f"anchored_{GROWTH_OBJ_NAME}"]   # objective chosen in Cell 4b

        # Seasonal naive: same ISO week one year earlier, matched on (unit, epi_week).
        prev = (M[(M.year == ty - 1)].groupby(["unit", "epi_week"])["cases"].mean()
                  .rename("snaive").reset_index())
        out = out.merge(prev, on=["unit", "epi_week"], how="left")
        out["snaive"] = out["snaive"].fillna(out["persistence"])

        # Neural comparator. Only built when this frame feeds the model shootout: the
        # ablation, resolution and latency cells read `anchored` alone, so fitting an
        # MLP inside each of them costs minutes per call and changes no reported number.
        # The tree models, the GLM and the linear model are all
        # different, but none of them is the architecture this literature actually
        # reaches for. A small MLP on the same features answers "you only tried
        # gradient boosting" without pretending a 6,000-row panel supports a deep
        # sequence model. Early stopping on an internal split, standardised inputs,
        # averaged over SEEDS like everything else.
        if with_extras:
          try:
            _med_n = tr[F].median(numeric_only=True)
            Xtr_n = tr[F].fillna(_med_n).fillna(0.0).to_numpy(float)
            Xte_n = te[F].fillna(_med_n).fillna(0.0).to_numpy(float)
            _mu, _sd = Xtr_n.mean(0), Xtr_n.std(0)
            _sd[_sd == 0] = 1.0
            Xtr_n = (Xtr_n - _mu) / _sd; Xte_n = (Xte_n - _mu) / _sd
            _ytr = np.log1p(tr[tl].to_numpy(float))
            _ps = []
            for sd in SEEDS:
                _mlp = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu",
                                    alpha=1e-3, learning_rate_init=1e-3, max_iter=300,
                                    early_stopping=True, n_iter_no_change=15,
                                    validation_fraction=0.15, random_state=sd)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    _mlp.fit(Xtr_n, _ytr)
                _ps.append(np.expm1(_mlp.predict(Xte_n)))
            out["mlp"] = np.clip(np.mean(_ps, axis=0), 0, None)
          except Exception as e:
            log.warning("MLP failed at h=%d, %s: %s", h, ty, e)
            out["mlp"] = out["persistence"]

        # Negative-binomial GLM. The audit's "only one model family" objection could
        # not be answered by ARIMA, whose order is from national monthly data and is
        # out of regime here. This is the standard count model for overdispersed
        # surveillance counts, given the same features as the trees, ridge-penalised
        # because IRLS on 77 correlated predictors does not otherwise converge.
        # Negative-binomial GLM, the standard count model for overdispersed
        # surveillance data. Specification matters more than it looks: an earlier
        # version used log(population) as an OFFSET, which forces cases to be
        # proportional to population and scored -172%. Conditional on last week's
        # count that constraint is simply wrong, and letting log-population enter as
        # a free covariate instead moves the same model to about -9%. Reporting the
        # offset version would have been a strawman baseline, so this one is given
        # the specification a statistician would actually choose: log1p on the
        # autoregressive terms, season, climate, log-population free, dispersion
        # estimated from a Poisson first pass, and a feature set small enough for
        # IRLS to converge rather than all 77 collinear columns.
        if with_extras:
          try:
            _glm_f = [c for c in (["cases_lag1", "cases_lag2", "cases_lag3", "cases_lag4"]
                                  + G_SEASON + CLIM_BASE + ["population"]) if c in tr.columns]
            def _glm_design(frame):
                X = frame[_glm_f].copy()
                for c in _glm_f:
                    if c.startswith("cases_lag") or c == "population":
                        X[c] = np.log1p(X[c].clip(lower=0))
                X = X.fillna(X.median(numeric_only=True)).fillna(0.0)
                return sm.add_constant(X, has_constant="add")

            Xg_tr, Xg_te = _glm_design(tr), _glm_design(te)
            _po = sm.GLM(tr[tl].to_numpy(float), Xg_tr,
                         family=sm.families.Poisson()).fit()
            _fv = np.asarray(_po.fittedvalues, float)
            _alpha = float(np.sum((tr[tl].to_numpy(float) - _fv) ** 2 - _fv) /
                           max(np.sum(_fv ** 2), 1e-9))
            _alpha = float(np.clip(_alpha, 1e-3, 10.0))
            _nb = sm.GLM(tr[tl].to_numpy(float), Xg_tr,
                         family=sm.families.NegativeBinomial(alpha=_alpha)).fit()
            out["nb_glm"] = np.clip(np.asarray(_nb.predict(Xg_te), float), 0, None)
          except Exception as e:                    # never let a baseline kill the run
            log.warning("negative-binomial GLM failed at h=%d, %s: %s", h, ty, e)
            out["nb_glm"] = out["persistence"]

        # Log-AR(4) ridge, a linear floor.
        ar = [c for c in ["log_cases_lag0", "log_cases_lag1", "log_cases_lag2", "log_cases_lag3"] if c in F]
        if ar:
            r = Ridge(alpha=10.0).fit(tr[ar], np.log1p(tr[tl]))
            out["ridge_ar4"] = np.clip(np.expm1(r.predict(te[ar])), 0, None)
        else:
            out["ridge_ar4"] = out["persistence"]

        frames.append(out)
    P = pd.concat(frames, ignore_index=True)
    if "nb_glm" in P.columns and P["nb_glm"].notna().any():
        _mp = mean_absolute_error(P.y, P.persistence)
        _mg = mean_absolute_error(P.y, P.nb_glm)
        if _mp > 0 and _mg / _mp > 3.0:
            log.warning("[gate] nb_glm MAE is %.1fx persistence at h=%d - that is a "
                        "misspecified model, not a finding, and must not be reported "
                        "as a baseline", _mg / _mp, h)
        _same = float(np.mean(np.isclose(P["nb_glm"].values, P["persistence"].values)))
        if _same > 0.99:
            log.warning("[gate] nb_glm is a copy of persistence in %.0f%% of rows at h=%d - "
                        "the GLM did not fit and must not be reported as a baseline",
                        100 * _same, h)
        else:
            log.info("  nb_glm fitted (differs from persistence in %.0f%% of rows)",
                     100 * (1 - _same))
    # ARIMA is independent of the feature set, so it is computed once per panel and
    # merged in; the ablation and resolution loops reuse the cache.
    if arima_tag:
        A = arima_baseline(M, h, test_years, tag=arima_tag)
        if len(A):
            P = P.merge(A, on=["unit", "week_start"], how="left")
            P["arima"] = P["arima"].fillna(P["persistence"])
        else:
            P["arima"] = P["persistence"]
    return P


PRED = {h: rolling_origin(MD, FD_FULL, h, arima_tag="district", with_extras=True)
        for h in HORIZONS}
log.info("rolling-origin predictions built for h = %s", HORIZONS)

SEED_COLS = lambda df: [c for c in df.columns if c.startswith("__seed")]

MODELS = ["persistence", "snaive", "arima", "ridge_ar4", "nb_glm", "mlp",
          "level_L2", "level_Poisson", "level_Tweedie",
          "anchored_L2", "anchored_L1"]
LABELS = {"persistence": "Lag-0 persistence", "snaive": "Seasonal naive",
          "arima": f"ARIMA{ARIMA_ORDER} per district (order from Naher 2022, national monthly)",
          "ridge_ar4": "Log-AR(4) ridge",
          "nb_glm": "Negative-binomial GLM (log-pop covariate, estimated dispersion)",
          "mlp": "MLP (64-32, early stopping)",
          "level_L2": "LightGBM level (L2)",
          "level_Poisson": "LightGBM level (Poisson)",
          "level_Tweedie": "LightGBM level (Tweedie)",
          "anchored_L2": "LightGBM anchored growth (L2)",
          "anchored_L1": "LightGBM anchored growth (L1)",
          "anchored": "LightGBM anchored growth (L1)"}

rows = []
for h in HORIZONS:
    P = PRED[h]; base = mean_absolute_error(P.y, P.persistence)
    for m in MODELS:
        if m not in P.columns:
            continue
        mm = metrics(P.y, P[m])
        rows.append({"horizon_weeks": h, "model": LABELS[m], "key": m,
                     "MAE": round(mm["MAE"], 3), "RMSE": round(mm["RMSE"], 3),
                     "wMAPE_pct": round(mm["wMAPE"], 2),
                     "skill_vs_persistence_pct": round(100 * (1 - mm["MAE"] / base), 2)})
shootout = save_table("table2_forecast_shootout", pd.DataFrame(rows),
                      f"Multi-horizon prospective forecast comparison, district "
                      f"resolution, FULL feature set ({len(FD_FULL)} features). Skill "
                      "numbers are only comparable across tables that use the same "
                      "feature set: table9 uses the shared set and is lower for that "
                      "reason alone. The ARIMA row uses a published order fitted at a "
                      "different resolution - see table0 before quoting it")
print(shootout.pivot_table(index="horizon_weeks", columns="model",
                           values="skill_vs_persistence_pct").round(1).to_string())

In [ ]:
# =============================================================================
# CELL 6 — Block bootstrap + Diebold–Mariano on the headline deltas
# =============================================================================
def block_bootstrap_delta(P, a, b, n_boot=2000, seed=SEED, level="unit"):
    """95% interval on MAE(a) - MAE(b), resampling whole blocks.

    level="unit"  : (district, fold) blocks - preserves serial correlation.
    level="block" : (division, fold) blocks - ALSO preserves the spatial correlation
                    between neighbouring districts, which the unit-level version
                    ignores. Districts in this panel are spatially autocorrelated
                    (Hossain 2024), so unit-level intervals are anti-conservative.
                    Only 8 divisions, so these intervals are wide and honest.
    """
    rng = np.random.default_rng(seed)
    key = ["unit", "fold"] if level == "unit" else ["block", "fold"]
    blocks = list(P.groupby(key).indices.values())
    y, pa, pb = P.y.values, P[a].values, P[b].values
    draws = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))])
        draws[i] = np.abs(y[idx] - pa[idx]).mean() - np.abs(y[idx] - pb[idx]).mean()
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


def panel_dm(P, a, b):
    """Panel Diebold-Mariano.

    Each unit contributes ONE number - its mean absolute-error differential - and the
    test is a one-sample t-test across units. Collapsing to the unit level absorbs the
    serial correlation that a multi-step forecast necessarily induces (an h-step error
    sequence is MA(h-1)), so no HAC correction is needed and the cross-sectional
    dependence is handled by treating units as the sampling unit.
    """
    d = (P.assign(_d=np.abs(P.y - P[a]) - np.abs(P.y - P[b]))
           .groupby("unit")["_d"].mean())
    if len(d) < 3 or d.std(ddof=1) == 0:
        return np.nan, np.nan, len(d)
    t, pv = stats.ttest_1samp(d.values, 0.0)
    return float(t), float(pv), int(len(d))


def bh_fdr(pvals, q=ALPHA):
    """Benjamini-Hochberg. Returns the boolean reject vector and adjusted p-values."""
    p = np.asarray(pvals, float)
    ok = np.isfinite(p)
    adj = np.full_like(p, np.nan)
    if ok.sum() == 0:
        return np.zeros_like(p, bool), adj
    idx = np.where(ok)[0][np.argsort(p[ok])]
    n = len(idx)
    prev = 1.0
    for rank in range(n - 1, -1, -1):
        prev = min(prev, p[idx[rank]] * n / (rank + 1))
        adj[idx[rank]] = prev
    return (adj <= q) & ok, adj


rows = []
for h in HORIZONS:
    P = PRED[h]
    for a, b in [("anchored_L1", "persistence"), ("level_Tweedie", "persistence"),
                 ("anchored_L1", "level_Tweedie"), ("anchored_L1", "anchored_L2"),
                 ("anchored_L1", "arima")]:
        if a not in P.columns or b not in P.columns:
            continue
        lo, hi = block_bootstrap_delta(P, a, b)
        slo, shi = block_bootstrap_delta(P, a, b, level="block")
        t, pv, nu = panel_dm(P, a, b)
        rows.append({"horizon_weeks": h, "comparison": f"{LABELS[a]} vs {LABELS[b]}",
                     "delta_MAE": round(mean_absolute_error(P.y, P[a]) - mean_absolute_error(P.y, P[b]), 3),
                     "boot_lo": round(lo, 3), "boot_hi": round(hi, 3),
                     "boot_excludes_0": bool(lo * hi > 0),
                     "spatial_boot_lo": round(slo, 3), "spatial_boot_hi": round(shi, 3),
                     "spatial_boot_excludes_0": bool(slo * shi > 0),
                     "DM_t": round(t, 3), "DM_p_raw": round(pv, 4), "n_units": nu})
unc = pd.DataFrame(rows)
# One family of tests, so the p-values are corrected together.
rej, adj = bh_fdr(unc["DM_p_raw"].values, q=ALPHA)
unc["DM_p_BH"] = np.round(adj, 4)

# No binary significant/not column. Comparing the previous version's Kaggle and
# local runs showed three of these labels flipping between platforms while the
# point estimates agreed to 0.29 MAE and all sixteen bootstrap intervals overlapped:
# the estimates are reproducible, a step function evaluated at its discontinuity is
# not. What is reported instead is the effect and its interval, plus which of the two
# tests each comparison satisfies, so the reader can see where they disagree.
unc["dm_reject_BH"] = rej
unc["evidence"] = np.where(rej & unc["boot_excludes_0"], "both tests",
                    np.where(rej, "DM only",
                      np.where(unc["boot_excludes_0"], "bootstrap only", "neither")))
# The spatially-blocked interval is the conservative one: it is the number to quote
# if any single interval is quoted, because districts are not independent.
unc["survives_spatial_blocking"] = unc["spatial_boot_excludes_0"]
uncert = save_table("table3_uncertainty_on_deltas", unc,
                    f"Block-bootstrap intervals and panel Diebold-Mariano tests, "
                    f"Benjamini-Hochberg corrected at q={ALPHA}. Effects and intervals, "
                    f"not significance labels - see the note in this cell")
print(uncert[["horizon_weeks", "comparison", "delta_MAE", "boot_lo", "boot_hi",
              "spatial_boot_lo", "spatial_boot_hi", "DM_p_BH", "evidence",
              "survives_spatial_blocking"]].to_string(index=False))
log.info("of %d comparisons, %d have unit-level intervals excluding zero but only %d "
         "survive spatial blocking - districts are not independent and the unit-level "
         "interval is anti-conservative",
         len(uncert), int(uncert.boot_excludes_0.sum()),
         int(uncert.survives_spatial_blocking.sum()))
_b = int(uncert.boot_excludes_0.sum()); _d = int(uncert.dm_reject_BH.sum())
_both = int((uncert.evidence == "both tests").sum())
log.info("evidence across %d comparisons: bootstrap excludes zero in %d, "
         "Diebold-Mariano rejects in %d, both agree in %d. The remainder are "
         "underpowered - two test seasons cannot resolve differences of this size.",
         len(uncert), _b, _d, _both)

In [ ]:
# =============================================================================
# CELL 7 — Prediction intervals: empirical baseline, split conformal, adaptive
# =============================================================================
def _cal_split(M, tg, ty, cal_weeks):
    """Split the training period into fit and conformal-calibration sets.

    Calibration weeks are spread EVENLY across the training period rather than
    taken as the trailing block. A trailing window on the 2024 fold lands entirely
    inside the second half of 2023 - the peak of the record epidemic - so the
    non-conformity scores describe a regime the test year does not contain. That
    produced a negative correction, shrinking already-narrow intervals: at h=4 one
    seed set gave raw 0.851 -> conformal 0.726. Even spacing keeps the calibration
    residuals representative of the whole training period while staying strictly
    earlier than the test year.
    """
    tra = M[(M.year < ty) & M[tg].notna()].sort_values("week_start")
    wk = np.sort(tra["week_start"].unique())
    if len(wk) <= cal_weeks + 4:
        return tra, tra
    idx = np.unique(np.linspace(0, len(wk) - 1, cal_weeks).round().astype(int))
    is_cal = tra["week_start"].isin(set(wk[idx]))
    return tra[~is_cal], tra[is_cal]


def wis(y, lo, hi, med, alpha=1 - NOMINAL):
    """Weighted interval score for a single central interval plus a median."""
    y, lo, hi, med = map(lambda v: np.asarray(v, float), (y, lo, hi, med))
    interval = (hi - lo) + (2 / alpha) * (lo - y) * (y < lo) + (2 / alpha) * (y - hi) * (y > hi)
    return float(np.mean((0.5 * np.abs(y - med) + (alpha / 2) * interval) / 1.5))


def _conformal_q(scores, level=NOMINAL):
    """Finite-sample conformal quantile."""
    n = len(scores)
    if n < 10:
        return float(np.max(scores)) if n else 0.0
    return float(np.quantile(scores, min(1.0, np.ceil((n + 1) * level) / n)))


def interval_run(M, F, h, method, test_years=TEST_YEARS, gamma=None, groups=None, space="growth"):
    """method in {'empirical','raw','split','mondrian','adaptive'}.

    'mondrian' calibrates a separate correction within each burden group, which is
    the fix for the conditional under-coverage that a single shared correction hides.
    """
    gamma = ACI_GAMMA if gamma is None else gamma
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    frames = []
    for ty in test_years:
        te = M[(M.year == ty) & M[tl].notna()].sort_values("week_start")
        if len(te) < 20:
            continue
        anchor = te["cases_lag0"].values + 1.0
        y = te[tl].values

        if method == "empirical":
            # No model: per-unit empirical quantiles of historical h-step log growth.
            tr = M[(M.year < ty) & M[tg].notna()]
            ql = tr.groupby("unit")[tg].quantile(0.05); qh = tr.groupby("unit")[tg].quantile(0.95)
            qm = tr.groupby("unit")[tg].median()
            gl = te["unit"].map(ql).fillna(tr[tg].quantile(0.05)).values
            gh = te["unit"].map(qh).fillna(tr[tg].quantile(0.95)).values
            gm = te["unit"].map(qm).fillna(tr[tg].median()).values
            lo, hi, med = anchor * np.exp(gl) - 1, anchor * np.exp(gh) - 1, anchor * np.exp(gm) - 1
        else:
            # `space` decides what the quantile models are fitted to. "growth" is the
            # anchored log-ratio the paper reports; "level" fits the counts directly.
            # The distinction matters for one specific objection: the anchored
            # back-transform (y_t+1)*exp(.)-1 is multiplicative in the anchor, so
            # interval WIDTH scales with district burden by construction. If the
            # conditional-coverage gradient is a property of conformal prediction on
            # epidemic counts it must also appear in level space; if it is an artifact
            # of the transform it will not. Cell 8c runs both and reports the answer.
            _tcol = tg if space == "growth" else tl
            tr, cal = _cal_split(M, _tcol, ty, CAL_WEEKS_BY_H.get(h, CAL_WEEKS))
            # Quantile loss is pinball; alpha=0.50 is the L1 median. Averaged over
            # SEEDS for the same reason the point forecasts are.
            _fitted = {a: [lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd},
                                             objective="quantile", alpha=a).fit(tr[F], tr[_tcol])
                           for sd in SEEDS] for a in (0.05, 0.50, 0.95)}
            q = {a: (lambda ms: (lambda X: np.mean([m.predict(X) for m in ms], axis=0)))(_fitted[a])
                 for a in (0.05, 0.50, 0.95)}
            # Normalised CQR. The raw score is divided by the model's own predicted
            # interval width, so the correction is a MULTIPLIER on that width rather
            # than a constant added in log-ratio space. An additive score calibrated
            # on high-variance weeks and applied to low-variance ones (or the reverse)
            # over- or under-corrects; a multiplicative one travels between regimes.
            c_lo, c_hi = q[0.05](cal[F]), q[0.95](cal[F])
            # Floor the denominator at the 5th percentile of observed calibration
            # widths rather than 1e-6. A near-degenerate predicted interval otherwise
            # divides the score into the thousands, which is numerically ugly even
            # though the 90th-percentile quantile survives it.
            w_floor = max(float(np.quantile(c_hi - c_lo, 0.05)), 1e-3)
            c_w = np.maximum(c_hi - c_lo, w_floor)
            s = np.maximum(c_lo - cal[_tcol].values, cal[_tcol].values - c_hi) / c_w
            g_lo, g_hi, g_md = (q[0.05](te[F]), q[0.95](te[F]), q[0.50](te[F]))
            t_w = np.maximum(g_hi - g_lo, w_floor)

            if method == "raw":
                qh_vec = np.zeros(len(te))
            elif method == "split":
                qh_vec = np.full(len(te), _conformal_q(s)) * t_w
            elif method == "mondrian":
                g_cal = cal["unit"].map(groups).fillna(-1).values
                g_te = te["unit"].map(groups).fillna(-1).values
                qh_vec = np.full(len(te), _conformal_q(s))       # fallback for unseen groups
                for gid in np.unique(g_te):
                    sel_c, sel_t = g_cal == gid, g_te == gid
                    if sel_c.sum() >= 30:
                        qh_vec[sel_t] = _conformal_q(s[sel_c])
                qh_vec = qh_vec * t_w
            else:
                # Adaptive conformal (Gibbs & Candes). 'adaptive' runs one shared
                # alpha_t; 'mondrian_adaptive' runs one per burden group, so it
                # corrects for drift AND for the heterogeneity Mondrian exposed.
                per_group = method == "mondrian_adaptive"
                g_cal = cal["unit"].map(groups).fillna(-1).values if per_group else np.zeros(len(cal))
                g_te = te["unit"].map(groups).fillna(-1).values if per_group else np.zeros(len(te))
                gids = np.unique(g_te)
                score_by_g = {gid: (s[g_cal == gid] if (g_cal == gid).sum() >= 30 else s)
                              for gid in gids}
                alpha_by_g = {gid: 1 - NOMINAL for gid in gids}
                qh_vec = np.empty(len(te))
                weeks = te["week_start"].values
                for wk in pd.unique(weeks):
                    for gid in gids:
                        sel = (weeks == wk) & (g_te == gid)
                        if not sel.any():
                            continue
                        lvl = float(np.clip(1 - alpha_by_g[gid], 0.01, 0.999))
                        _o = np.quantile(score_by_g[gid], lvl) * t_w[sel]
                        qh_vec[sel] = _o
                        if space == "growth":
                            _l = anchor[sel] * np.exp(g_lo[sel] - _o) - 1
                            _h = anchor[sel] * np.exp(g_hi[sel] + _o) - 1
                        else:
                            _l, _h = g_lo[sel] - _o, g_hi[sel] + _o
                        cov_wk = ((y[sel] >= _l) & (y[sel] <= _h)).mean()
                        alpha_by_g[gid] += gamma * ((1 - NOMINAL) - (1 - cov_wk))
            if space == "growth":
                lo = anchor * np.exp(g_lo - qh_vec) - 1
                hi = anchor * np.exp(g_hi + qh_vec) - 1
                med = anchor * np.exp(g_md) - 1
            else:
                lo, hi, med = g_lo - qh_vec, g_hi + qh_vec, g_md

        f = te[["unit", "block", "year", "epi_week", "week_start"]].copy()
        f["y"] = y; f["lo"] = np.clip(lo, 0, None); f["hi"] = hi
        f["med"] = np.clip(med, 0, None); f["fold"] = ty
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


BURDEN_G = burden_groups(MD)
INT_METHODS = ["empirical", "raw", "split", "mondrian", "adaptive", "mondrian_adaptive"]
INT_LABEL = {"empirical": "Empirical growth quantiles (no model)",
             "raw": "Quantile LightGBM (uncalibrated)",
             "split": "Split-conformal CQR",
             "mondrian": "Group-conditional conformal (Mondrian)",
             "adaptive": "Adaptive conformal (ACI)",
             "mondrian_adaptive": "Group-conditional adaptive conformal"}

INTERVALS = {}
rows = []
for h in [1, 2, 4]:
    for meth in INT_METHODS:
        I = interval_run(MD, FD_FULL, h, meth, groups=BURDEN_G)
        INTERVALS[(h, meth)] = I
        cov = float(((I.y >= I.lo) & (I.y <= I.hi)).mean())
        rows.append({"horizon_weeks": h, "method": INT_LABEL[meth],
                     "key": meth, "nominal": NOMINAL,
                     "empirical_coverage": round(cov, 4),
                     "median_width_cases": round(float(np.median(I.hi - I.lo)), 2),
                     "mean_width_cases": round(float(np.mean(I.hi - I.lo)), 2),
                     "WIS": round(wis(I.y, I.lo, I.hi, I.med), 3)})
def coverage_test(I, nominal=NOMINAL):
    """Is empirical coverage different from nominal? Unit-level coverage rates, t-test.

    Each district contributes one coverage proportion, so within-district dependence
    does not inflate the test the way a row-level binomial test would.
    """
    cov = (I.assign(_c=((I.y >= I.lo) & (I.y <= I.hi)).astype(float))
             .groupby("unit")["_c"].mean())
    t, pv = stats.ttest_1samp(cov.values, nominal)
    return float(cov.mean()), float(t), float(pv), int(len(cov))

for r in rows:
    I = INTERVALS[(r["horizon_weeks"], r["key"])]
    _, t, pv, nu = coverage_test(I)
    r["cov_t"] = round(t, 3); r["cov_p_raw"] = round(pv, 4); r["n_units"] = nu
_c = pd.DataFrame(rows)
_rej, _adj = bh_fdr(_c["cov_p_raw"].values, q=ALPHA)
_c["cov_p_BH"] = np.round(_adj, 4)
# For a calibration claim the DESIRABLE outcome is failing to reject: coverage
# indistinguishable from nominal. Stating it this way round avoids reading a
# non-significant result as evidence of nothing.
_c["verdict"] = np.where(_rej, f"differs from {NOMINAL:.2f} (BH q<{ALPHA})",
                         f"indistinguishable from {NOMINAL:.2f}")
rows = _c.to_dict("records")
calib = save_table("table4_interval_calibration", _c,
                   "Coverage, width, WIS, and a test of coverage against nominal")
print(calib.pivot_table(index="horizon_weeks", columns="key",
                        values="empirical_coverage").round(3).to_string())

In [ ]:
# =============================================================================
# CELL 8 — Conditional coverage by district burden. Marginal coverage hides this.
# =============================================================================
# Reuse BURDEN_G rather than re-deriving the tertiles. The two definitions differed by
# one year of history and moved 2 of 64 districts across a boundary, which meant this
# table was scoring the group-conditional method against groups it was not calibrated
# on. Small, but it is the paper's lead result and it has to be the same grouping.
_TERT_NAME = {0: "low burden", 1: "mid burden", 2: "high burden"}
_order = ["low burden", "mid burden", "high burden"]
tert = pd.Series({u: _TERT_NAME[g] for u, g in BURDEN_G.items()}, name="tertile").astype(
    pd.CategoricalDtype(_order, ordered=True))   # keep table rows in burden order
assert set(tert.index) >= set(MD["unit"].unique()), "a modelled district has no burden group"

rows = []
for h in [1, 2, 4]:
    for meth in ["raw", "split", "mondrian", "adaptive", "mondrian_adaptive"]:
        I = INTERVALS[(h, meth)].copy()
        I["tertile"] = I["unit"].map(tert)
        for t, g in I.groupby("tertile", observed=True):
            rows.append({"horizon_weeks": h, "method": INT_LABEL[meth], "burden_tertile": str(t),
                         "n_unit_weeks": len(g),
                         "coverage": round(float(((g.y >= g.lo) & (g.y <= g.hi)).mean()), 4),
                         "median_width_cases": round(float(np.median(g.hi - g.lo)), 2)})
cond = save_table("table5_conditional_coverage", pd.DataFrame(rows),
                  "Coverage conditional on district burden tertile")
print(cond[cond.horizon_weeks == 2].to_string(index=False))

In [ ]:
# =============================================================================
# CELL 8b — Is the conditional-coverage difference real, and is it burden?
# =============================================================================
# The audit found that the paper's lead result was the only untested one: the
# spread of coverage across burden tertiles was reported descriptively. Two
# questions are settled here. (a) Is the spread significantly non-zero for each
# method, and is the reduction between methods significant? (b) Is "burden" the
# right explanation, or a proxy for district size / reporting intensity?

def _two_prop(k1, n1, k2, n2):
    """Two-proportion z-test with a Wald CI on the difference."""
    p1, p2 = k1 / n1, k2 / n2
    pp = (k1 + k2) / (n1 + n2)
    se0 = np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se0 if se0 > 0 else np.nan
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    return float(p1 - p2), float(z), float(2 * (1 - stats.norm.cdf(abs(z)))), \
           float(p1 - p2 - 1.96 * se), float(p1 - p2 + 1.96 * se)


def _spread_boot(I, groups, n_boot=2000, seed=SEED):
    """Bootstrap the max-min coverage spread across groups, resampling DISTRICTS."""
    rng = np.random.default_rng(seed)
    I = I.assign(_g=I["unit"].map(groups), _c=((I.y >= I.lo) & (I.y <= I.hi)).astype(float))
    units = I["unit"].unique()
    idx = {u: I.index[I.unit == u].values for u in units}
    out = []
    for _ in range(n_boot):
        pick = np.concatenate([idx[u] for u in rng.choice(units, len(units), replace=True)])
        gg = I.loc[pick].groupby("_g")["_c"].mean()
        if len(gg) >= 2:
            out.append(gg.max() - gg.min())
    return np.asarray(out)


cc_rows, sp_rows = [], []
for h in [1, 2, 4]:
    for meth in ["raw", "split", "mondrian_adaptive"]:
        I = INTERVALS[(h, meth)].copy()
        I["_g"] = I["unit"].map(BURDEN_G)
        cov = I.assign(_c=((I.y >= I.lo) & (I.y <= I.hi)).astype(int)).groupby("_g")["_c"]
        agg = cov.agg(["sum", "count"])
        if {0, 2} <= set(agg.index):
            dlt, z, pv, lo_, hi_ = _two_prop(agg.loc[0, "sum"], agg.loc[0, "count"],
                                             agg.loc[2, "sum"], agg.loc[2, "count"])
            cc_rows.append({"horizon_weeks": h, "method": INT_LABEL[meth],
                            "coverage_low_burden": round(agg.loc[0, "sum"] / agg.loc[0, "count"], 4),
                            "coverage_high_burden": round(agg.loc[2, "sum"] / agg.loc[2, "count"], 4),
                            "difference": round(dlt, 4), "ci_lo": round(lo_, 4),
                            "ci_hi": round(hi_, 4), "z": round(z, 3), "p_raw": round(pv, 6)})
    # is the SPREAD significantly smaller under the group-conditional method?
    a = _spread_boot(INTERVALS[(h, "split")], BURDEN_G)
    b = _spread_boot(INTERVALS[(h, "mondrian_adaptive")], BURDEN_G)
    diff = a - b
    sp_rows.append({"horizon_weeks": h,
                    "spread_split": round(float(np.mean(a)), 4),
                    "spread_group_conditional": round(float(np.mean(b)), 4),
                    "reduction": round(float(np.mean(diff)), 4),
                    "ci_lo": round(float(np.percentile(diff, 2.5)), 4),
                    "ci_hi": round(float(np.percentile(diff, 97.5)), 4),
                    "boot_p": round(float(2 * min((diff <= 0).mean(), (diff >= 0).mean())), 4)})

CC = pd.DataFrame(cc_rows)
if len(CC):
    _rej, _adj = bh_fdr(CC["p_raw"].values, q=ALPHA)
    CC["p_BH"] = np.round(_adj, 6)
    CC["verdict"] = np.where(_rej, f"coverage differs by burden (BH q<{ALPHA})",
                             "no detectable difference by burden")
save_table("table5b_conditional_coverage_test", CC,
           "Two-proportion test of low- vs high-burden coverage, per method")
print(CC.to_string(index=False))
SPR = save_table("table5c_spread_reduction", pd.DataFrame(sp_rows),
                 "Bootstrap of the coverage-spread reduction from group-conditional "
                 "calibration, resampling districts")
print(SPR.to_string(index=False))

# --- (b) burden, or a proxy for something else? -------------------------------
# Per-district coverage regressed on burden and on the size covariates it might be
# standing in for. If burden survives and population does not, "big districts are
# harder" is dead as an explanation.
I2 = INTERVALS[(2, "split")].copy()
I2["_c"] = ((I2.y >= I2.lo) & (I2.y <= I2.hi)).astype(float)
per_unit = I2.groupby("unit")["_c"].mean().rename("coverage").reset_index()
# No silent fallback here. Substituting a row count for a missing population column
# would still produce a plausible correlation, labelled "population", and the claim
# that district size is ruled out would rest on a variable that is not population.
_need = ["population", "pop_density"]
_miss = [c for c in _need if c not in MD.columns]
assert not _miss, (f"confound regression needs {_miss}, which the panel does not carry - "
                   "without them the size explanation cannot be tested and must not be "
                   "reported as ruled out")
cov_src = MD[(MD.year >= ALARM_BASE_MIN) & (MD.year <= ALARM_TRAIN_MAX)].groupby("unit").agg(
    burden=("cases", "mean"),
    population=("population", "first"),
    pop_density=("pop_density", "first"),
).reset_index()
assert cov_src["population"].nunique() > 1, "population is constant across districts"
reg = per_unit.merge(cov_src, on="unit")
reg["log_burden"] = np.log1p(reg["burden"])
reg["log_population"] = np.log1p(reg["population"])
rows = []
for v in ["log_burden", "log_population", "pop_density"]:
    if reg[v].nunique() < 3:
        continue
    r, pv = stats.spearmanr(reg[v], reg["coverage"])
    rows.append({"covariate": v, "spearman_rho": round(float(r), 4),
                 "p_value": round(float(pv), 6), "n_districts": len(reg)})
# Marginal correlations cannot separate burden from its correlates. Population COUNT
# is not associated, but population DENSITY is, and density is itself a burden
# correlate - so "coverage degrades with burden, not with district size" is only
# supportable if burden survives controlling for density. Partial Spearman does that:
# rank-transform, residualise each variable on the control by OLS, correlate the
# residuals. Reported for both directions so neither is privileged.
def _partial_spearman(df, x, y, ctrl):
    r = df[[x, y, ctrl]].rank()
    c = np.c_[np.ones(len(r)), r[ctrl].values]
    res = {}
    for v in (x, y):
        beta, *_ = np.linalg.lstsq(c, r[v].values, rcond=None)
        res[v] = r[v].values - c @ beta
    rho = float(stats.pearsonr(res[x], res[y])[0])
    # pearsonr's own p-value assumes df = n - 2, but one df went on the control, so the
    # test is df = n - 2 - k with k = 1. At n = 64 that is 61 rather than 62 - small, but
    # this is a number that goes in a paper, so it should be the right test.
    dof = len(r) - 2 - 1
    if dof <= 0 or abs(rho) >= 1:
        return rho, np.nan
    t = rho * np.sqrt(dof / (1 - rho ** 2))
    return rho, float(2 * stats.t.sf(abs(t), dof))


for _v, _c in [("log_burden", "pop_density"), ("pop_density", "log_burden")]:
    if reg[_v].nunique() < 3 or reg[_c].nunique() < 3:
        continue
    _r, _p = _partial_spearman(reg, _v, "coverage", _c)
    rows.append({"covariate": f"{_v} | controlling for {_c}", "spearman_rho": round(_r, 4),
                 "p_value": round(_p, 6), "n_districts": len(reg)})

BURD = pd.DataFrame(rows)
if len(BURD):
    _rej, _adj = bh_fdr(BURD["p_value"].values, q=ALPHA)
    BURD["p_BH"] = np.round(_adj, 6)
    BURD["verdict"] = np.where(_rej, "associated with coverage", "not associated")
save_table("table5d_coverage_covariates", BURD,
           "Per-district coverage under split conformal against burden and the size "
           "covariates burden might be proxying for. The last two rows are partial "
           "correlations: burden controlling for density, and density controlling for "
           "burden. Only the one that survives its control can be called the mechanism")
print(BURD.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 8c — Is the burden gradient conformal prediction, or our reparameterisation?
# =============================================================================
# The audit's strongest alternative explanation for the paper's lead result: every
# interval in Cell 7 is built on the anchored log-ratio, whose back-transform is
# multiplicative in the anchor, so absolute width scales with burden by construction.
# A coverage gradient across burden tertiles is therefore consistent with BOTH "a
# property of conformal prediction on epidemic counts" AND "an artifact of this
# transform". Rebuilding the same intervals in level space separates them.
lvl_rows = []
for h in [1, 2, 4]:
    for meth in ["split", "mondrian_adaptive"]:
        for space in ["growth", "level"]:
            I = (INTERVALS[(h, meth)] if space == "growth"
                 else interval_run(MD, FD_FULL, h, meth, groups=BURDEN_G, space="level"))
            I = I.copy()
            I["_g"] = I["unit"].map(BURDEN_G)
            cov = I.assign(_c=((I.y >= I.lo) & (I.y <= I.hi)).astype(int)).groupby("_g")["_c"]
            agg = cov.agg(["sum", "count"])
            if not {0, 2} <= set(agg.index):
                continue
            dlt, z, pv, lo_, hi_ = _two_prop(agg.loc[0, "sum"], agg.loc[0, "count"],
                                             agg.loc[2, "sum"], agg.loc[2, "count"])
            lvl_rows.append({
                "horizon_weeks": h, "method": INT_LABEL[meth], "target_space": space,
                "coverage_overall": round(float(((I.y >= I.lo) & (I.y <= I.hi)).mean()), 4),
                "coverage_low_burden": round(agg.loc[0, "sum"] / agg.loc[0, "count"], 4),
                "coverage_high_burden": round(agg.loc[2, "sum"] / agg.loc[2, "count"], 4),
                "burden_gap": round(dlt, 4), "ci_lo": round(lo_, 4), "ci_hi": round(hi_, 4),
                "z": round(z, 3), "p_raw": round(pv, 6),
                "median_width_cases": round(float(np.median(I.hi - I.lo)), 2)})

LVL = pd.DataFrame(lvl_rows)
if len(LVL):
    _rej, _adj = bh_fdr(LVL["p_raw"].values, q=ALPHA)
    LVL["p_BH"] = np.round(_adj, 6)
    LVL["verdict"] = np.where(_rej, "gradient present", "no detectable gradient")
save_table("table5e_gradient_parameterisation", LVL,
           "The burden gradient rebuilt in level space. If it appears under BOTH target "
           "parameterisations the finding generalises; if it appears only under the "
           "anchored log-ratio it must be scoped to that choice and cannot be described "
           "as a property of conformal prediction on epidemic counts")
print(LVL.to_string(index=False))
if len(LVL):
    _sp = LVL[(LVL.target_space == "level") & (LVL.method == INT_LABEL["split"])]
    if len(_sp):
        log.info("[audit] split-conformal burden gap in LEVEL space: %s (growth space: %s)",
                 list(_sp.burden_gap.round(4)),
                 list(LVL[(LVL.target_space == "growth") &
                          (LVL.method == INT_LABEL["split"])].burden_gap.round(4)))

In [ ]:
# =============================================================================
# CELL 9 — 2x2 space-time validation matrix, spatial folds rotated over all blocks
# =============================================================================
H_ALARM = 2
A = MD[MD[f"target_alarm_{H_ALARM}w"].notna()].copy()
YA = f"target_alarm_{H_ALARM}w"
log.info("alarm task: n=%s, base rate %.3f", f"{len(A):,}", A[YA].mean())

def _fit(tr, te, F=FD_FULL):
    """Seed-ensembled alarm classifier, matching the regressors."""
    p = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                 .fit(tr[F], tr[YA].astype(int)).predict_proba(te[F])[:, 1] for sd in SEEDS], axis=0)
    return p, te[YA].astype(int).values

def _pool(pairs):
    ps, ys = zip(*pairs)
    p, y = np.concatenate(ps), np.concatenate(ys)
    return float(roc_auc_score(y, p)), float(average_precision_score(y, p))

BLOCKS = sorted(A["block"].unique())
tr_c1, te_c1 = train_test_split(A, test_size=0.20, random_state=SEED, stratify=A[YA])
c1 = _pool([_fit(tr_c1, te_c1)])
c2 = _pool([_fit(A[A.year < ty], A[A.year == ty]) for ty in TEST_YEARS])
c3 = _pool([_fit(A[A.block != b], A[A.block == b]) for b in BLOCKS])
c4 = _pool([_fit(A[(A.year < ty) & (A.block != b)], A[(A.year == ty) & (A.block == b)])
            for ty in TEST_YEARS for b in BLOCKS])

matrix = save_table("table6_optimism_gap", pd.DataFrame([
    {"condition": "C1 random 80/20", "holds_out": "nothing meaningful", "ROC_AUC": round(c1[0], 4), "PR_AUC": round(c1[1], 4)},
    {"condition": "C2 rolling origin", "holds_out": "future seasons", "ROC_AUC": round(c2[0], 4), "PR_AUC": round(c2[1], 4)},
    {"condition": "C3 leave-one-division-out", "holds_out": "unseen geography", "ROC_AUC": round(c3[0], 4), "PR_AUC": round(c3[1], 4)},
    {"condition": "C4 space + time", "holds_out": "both", "ROC_AUC": round(c4[0], 4), "PR_AUC": round(c4[1], 4)},
    {"condition": "OPTIMISM GAP (C1 - C4)", "holds_out": "", "ROC_AUC": round(c1[0] - c4[0], 4), "PR_AUC": round(c1[1] - c4[1], 4)},
]), "2x2 space-time validation matrix")
print(matrix.to_string(index=False))

assert c4[0] <= min(c2[0], c3[0]) + 0.01, "validation matrix incoherent: C4 above C2 or C3"
log.info("[gate] validation matrix ordering OK")

In [ ]:
# =============================================================================
# CELL 9b — Does the optimism gap survive a different outbreak threshold?
# =============================================================================
# The whole alarm arm, the optimism gap included, rests on one unjustified
# constant: the per-district 80th percentile of training-year cases. If the gap
# only exists at that threshold it is an artefact of the definition rather than a
# property of the validation design. The sweep below re-runs the full 2x2 matrix
# at four thresholds, holding everything else fixed.

ALARM_Q_SWEEP = [0.70, 0.75, 0.80, 0.85]
gap_rows = []

for _q in ALARM_Q_SWEEP:
    _Mq = add_alarm_labels(MD, quantile=_q)          # one definition, shared with the matrix
    _nfl = _Mq.attrs.get("n_floored", 0)
    _lab = _Mq[f"target_alarm_{H_ALARM}w"]
    Aq = _Mq.loc[_lab.notna()].copy()
    Aq["_y"] = _lab.loc[_lab.notna()].astype(int).values
    if Aq["_y"].nunique() < 2:
        continue

    def _fq(tr, te):
        pr = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                      .fit(tr[FD_FULL], tr["_y"]).predict_proba(te[FD_FULL])[:, 1]
                      for sd in SEEDS], axis=0)
        return pr, te["_y"].values

    def _pl(pairs):
        ps, ys = zip(*pairs)
        pp, yy = np.concatenate(ps), np.concatenate(ys)
        return float(roc_auc_score(yy, pp)), float(average_precision_score(yy, pp))

    _blocks = sorted(Aq["block"].unique())
    _tr1, _te1 = train_test_split(Aq, test_size=0.20, random_state=SEED, stratify=Aq["_y"])
    q1 = _pl([_fq(_tr1, _te1)])
    q2 = _pl([_fq(Aq[Aq.year < ty], Aq[Aq.year == ty]) for ty in TEST_YEARS])
    q3 = _pl([_fq(Aq[Aq.block != b], Aq[Aq.block == b]) for b in _blocks])
    q4 = _pl([_fq(Aq[(Aq.year < ty) & (Aq.block != b)], Aq[(Aq.year == ty) & (Aq.block == b)])
              for ty in TEST_YEARS for b in _blocks
              if Aq[(Aq.year == ty) & (Aq.block == b)]["_y"].nunique() > 1])

    gap_rows.append({
        "alarm_quantile": _q, "alarm_floor": ALARM_MIN_CASES,
        "base_rate": round(float(Aq["_y"].mean()), 4),
        "districts_at_floor": _nfl, "n_rows": len(Aq),
        "C1_ROC": round(q1[0], 4), "C2_ROC": round(q2[0], 4),
        "C3_ROC": round(q3[0], 4), "C4_ROC": round(q4[0], 4),
        "gap_ROC": round(q1[0] - q4[0], 4),
        "C1_PR": round(q1[1], 4), "C4_PR": round(q4[1], 4),
        "gap_PR": round(q1[1] - q4[1], 4),
        "ordering_holds": bool(q1[0] > q3[0] > q2[0] > q4[0])})

# A sensitivity sweep that cannot reproduce the value it is perturbing is measuring
# something else. This caught exactly that: the sweep used the modelling window while
# engineer() used a window including the 2019 partial season.
_base_row = [r for r in gap_rows if abs(r["alarm_quantile"] - ALARM_QUANTILE) < 1e-9
             and r["alarm_floor"] == ALARM_MIN_CASES]
if _base_row:
    _d = abs(_base_row[0]["gap_ROC"] - (c1[0] - c4[0]))
    assert _d < 0.01, (
        f"sensitivity sweep does not reproduce the reported gap at q={ALARM_QUANTILE} "
        f"(sweep {_base_row[0]['gap_ROC']:.4f} vs matrix {c1[0] - c4[0]:.4f}) - it is "
        "perturbing a different baseline and cannot be interpreted")
    log.info("[gate] sweep reproduces the reported gap at q=%.2f (delta %.4f)",
             ALARM_QUANTILE, _d)

# The floor is its own free parameter. Sweep it at the reported quantile so the
# gap is not resting on two unexamined constants instead of one.
def _plf(pairs):
    ps, ys = zip(*pairs)
    pp, yy = np.concatenate(ps), np.concatenate(ys)
    return float(roc_auc_score(yy, pp)), float(average_precision_score(yy, pp))

for _fl in [1, 3, 5, 10]:
    _Mf = add_alarm_labels(MD, quantile=ALARM_QUANTILE, floor=_fl)
    _labf = _Mf[f"target_alarm_{H_ALARM}w"]
    Af = _Mf.loc[_labf.notna()].copy()
    Af["_y"] = _labf.loc[_labf.notna()].astype(int).values
    if Af["_y"].nunique() < 2:
        continue

    def _ff(tr, te):
        pr = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                      .fit(tr[FD_FULL], tr["_y"]).predict_proba(te[FD_FULL])[:, 1]
                      for sd in SEEDS], axis=0)
        return pr, te["_y"].values

    _bl = sorted(Af["block"].unique())
    _t1, _e1 = train_test_split(Af, test_size=0.20, random_state=SEED, stratify=Af["_y"])
    f1 = _plf([_ff(_t1, _e1)])
    f4 = _plf([_ff(Af[(Af.year < ty) & (Af.block != b)], Af[(Af.year == ty) & (Af.block == b)])
              for ty in TEST_YEARS for b in _bl
              if Af[(Af.year == ty) & (Af.block == b)]["_y"].nunique() > 1])
    gap_rows.append({
        "alarm_quantile": ALARM_QUANTILE, "alarm_floor": _fl,
        "base_rate": round(float(Af["_y"].mean()), 4),
        "districts_at_floor": _Mf.attrs.get("n_floored", 0), "n_rows": len(Af),
        "C1_ROC": round(f1[0], 4), "C2_ROC": np.nan, "C3_ROC": np.nan,
        "C4_ROC": round(f4[0], 4), "gap_ROC": round(f1[0] - f4[0], 4),
        "C1_PR": round(f1[1], 4), "C4_PR": round(f4[1], 4),
        "gap_PR": round(f1[1] - f4[1], 4), "ordering_holds": np.nan})

# The floor sweep re-runs the reported configuration at floor=5, so it must land on
# the quantile sweep's q=0.80 row exactly. A free replication check on the whole path.
_dup = [r for r in gap_rows
        if abs(r["alarm_quantile"] - ALARM_QUANTILE) < 1e-9 and r["alarm_floor"] == ALARM_MIN_CASES]
if len(_dup) == 2:
    assert abs(_dup[0]["gap_ROC"] - _dup[1]["gap_ROC"]) < 1e-9, (
        f"the same configuration scored differently in the two sweeps "
        f"({_dup[0]['gap_ROC']:.4f} vs {_dup[1]['gap_ROC']:.4f}) - the alarm path is "
        "not deterministic and no sensitivity row can be trusted")
    log.info("[gate] quantile and floor sweeps agree exactly at the reported setting")

GAPS = save_table("table6b_alarm_threshold_sensitivity", pd.DataFrame(gap_rows),
                  "Optimism gap re-measured across outbreak quantiles (floor held at "
                  f"{ALARM_MIN_CASES}) and across floors (quantile held at "
                  f"{ALARM_QUANTILE}). districts_at_floor says how many of the 64 use the "
                  "floor rather than their own quantile - at low quantiles that is most of "
                  "them, so those rows are not pure quantile thresholds")
print(GAPS.to_string(index=False))
if len(GAPS):
    log.info("optimism gap across thresholds %s: ROC %.4f-%.4f, PR %.4f-%.4f; "
             "C1>C3>C2>C4 ordering holds in %d of %d",
             ALARM_Q_SWEEP, GAPS.gap_ROC.min(), GAPS.gap_ROC.max(),
             GAPS.gap_PR.min(), GAPS.gap_PR.max(),
             int(GAPS.ordering_holds.sum()), len(GAPS))

In [ ]:
# =============================================================================
# CELL 10 — Operational alarms: fixed sensitivity, false-alarm rate, lead time
# =============================================================================
alarm_rows, lead_rows, div_rows = [], [], []
for h in HORIZONS:
    ycol = f"target_alarm_{h}w"
    sub = MD[MD[ycol].notna()]
    tr = sub[sub.year <= 2023]
    te = sub[sub.year.isin(TEST_YEARS)].copy()
    te["p"] = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                       .fit(tr[FD_FULL], tr[ycol].astype(int))
                       .predict_proba(te[FD_FULL])[:, 1] for sd in SEEDS], axis=0)
    y, p = te[ycol].astype(int).values, te["p"].values

    # The operating threshold must be chosen WITHOUT the test labels. Reading it off
    # the test precision-recall curve and then scoring the same rows at it guarantees
    # the target sensitivity is hit exactly - the earlier version reported 0.800 and
    # 0.901 for that reason, and every precision and false-alarm figure with it was
    # optimistic. Here the threshold is fitted on the inner validation year (model
    # trained on <= SEL_TRAIN, threshold picked on SEL_VALID), then frozen and applied
    # unchanged to the test years. Achieved sensitivity is now an OUTCOME, not a target.
    tr_in = sub[sub.year <= SEL_TRAIN]
    va_in = sub[sub.year == SEL_VALID]
    thr_by_target = {}
    if len(va_in) and va_in[ycol].nunique() > 1:
        p_va = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                        .fit(tr_in[FD_FULL], tr_in[ycol].astype(int))
                        .predict_proba(va_in[FD_FULL])[:, 1] for sd in SEEDS], axis=0)
        y_va = va_in[ycol].astype(int).values
        pv, rv, tv = precision_recall_curve(y_va, p_va)
        for target in (0.80, 0.90):
            ok_v = np.where(rv[:-1] >= target)[0]
            if len(ok_v):
                thr_by_target[target] = float(tv[ok_v[-1]])

    # A sensitivity target fitted on one season does not transfer to another when the
    # base rate moves: 2023 ran a 0.372 alarm rate against 0.143 in the test years, so a
    # threshold permissive enough for 80% sensitivity there fires almost every week here.
    # That is a real finding about deployment and it is reported below, but it is not a
    # usable operating point. The alert-budget rule is: each week, rank the districts by
    # predicted score and alert the top k%. It uses no labels, needs no threshold to
    # transfer, and matches how a surveillance unit with fixed field capacity actually
    # works. Sensitivity and precision are then outcomes of a capacity decision.
    _wk_rank = te.groupby("week_start")["p"].rank(ascending=False, method="first")
    _wk_n = te.groupby("week_start")["p"].transform("size")
    for k_pct in (0.05, 0.10, 0.20):
        _cap = np.maximum(1, np.round(k_pct * _wk_n))
        predb = (_wk_rank <= _cap).astype(int).values
        tp = int(((predb == 1) & (y == 1)).sum()); fn = int(((predb == 0) & (y == 1)).sum())
        fp = int(((predb == 1) & (y == 0)).sum()); tn = int(((predb == 0) & (y == 0)).sum())
        alarm_rows.append({
            "horizon_weeks": h, "rule": f"alert budget, top {int(k_pct * 100)}% of districts",
            "target_sensitivity": np.nan,
            "achieved_sensitivity": round(tp / (tp + fn), 3) if tp + fn else np.nan,
            "precision": round(tp / (tp + fp), 3) if tp + fp else np.nan,
            "false_alarm_rate": round(fp / (fp + tn), 4) if fp + tn else np.nan,
            "alerts_per_week": round(float(_cap.mean()), 1),
            "threshold": np.nan, "threshold_source": "no labels used"})
        if h == H_ALARM:
            te[f"_alert_budget{int(k_pct * 100)}"] = predb

    for target in (0.80, 0.90):
        if target not in thr_by_target:
            continue
        t = thr_by_target[target]
        pred = (p >= t).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
        fp = int(((pred == 1) & (y == 0)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
        alarm_rows.append({"horizon_weeks": h,
                           "rule": f"sensitivity target fitted on {SEL_VALID}",
                           "target_sensitivity": target,
                           "achieved_sensitivity": round(tp / (tp + fn), 3) if tp + fn else np.nan,
                           "precision": round(tp / (tp + fp), 3) if tp + fp else np.nan,
                           "false_alarm_rate": round(fp / (fp + tn), 4) if fp + tn else np.nan,
                           "alerts_per_week": round(float(pred.sum()) / te["week_start"].nunique(), 1),
                           "threshold": round(float(t), 4),
                           "threshold_source": f"fitted on {SEL_VALID}, frozen"})
        if h == H_ALARM:
            te[f"_alert_sens{int(target * 100)}"] = pred
    # Lead time is NOT reported as a point estimate. Measured as "weeks from the first
    # alarm to the season's first threshold crossing", it is an artifact of whichever
    # operating rule is chosen: a threshold tuned on the test set gave 7 weeks, one
    # frozen on 2023 gave 16 (it fires continuously), and a weekly top-10% budget gave
    # 30 (it always alerts somebody, including in January). Any rule that is ever active
    # during the quiet season produces an arbitrarily long "lead". Rather than search for
    # the definition that yields the most attractive number, every rule is reported and
    # the spread is the finding: this design does not identify lead time.
    if h == H_ALARM:
        for _rule_col in [c for c in te.columns if c.startswith("_alert_")]:
            _leads = []
            for (u, yr), g in te.groupby(["unit", "year"]):
                g = g.sort_values("week_start")
                crossed = g[g["cases_lag0"] >= g["alarm_threshold"]]
                if not len(crossed):
                    continue
                onset = crossed["week_start"].min()
                fired = g[(g[_rule_col] == 1) & (g.week_start < onset) &
                          (g["cases_lag0"] < g["alarm_threshold"])]
                if len(fired):
                    _leads.append((onset - fired["week_start"].min()).days / 7)
            if _leads:
                _a = np.asarray(_leads, float)
                lead_rows.append({
                    "operating_rule": _rule_col.replace("_alert_budget", "top ")
                                               .replace("_alert_sens", "sensitivity target ")
                                      + ("% of districts per week"
                                         if "budget" in _rule_col else "% fitted on 2023"),
                    "n_district_seasons_with_prior_alarm": len(_a),
                    "median_lead_weeks": round(float(np.median(_a)), 1),
                    "iqr_lo": round(float(np.percentile(_a, 25)), 1),
                    "iqr_hi": round(float(np.percentile(_a, 75)), 1),
                    "max_lead_weeks": round(float(_a.max()), 1)})

    if h == H_ALARM:
            for b, g in te.groupby("block"):
                if g[ycol].nunique() > 1:
                    div_rows.append({"division": b, "n_unit_weeks": len(g),
                                     "outbreak_rate": round(float(g[ycol].mean()), 3),
                                     "ROC_AUC": round(float(roc_auc_score(g[ycol], g.p)), 4),
                                     "PR_AUC": round(float(average_precision_score(g[ycol], g.p)), 4)})

save_table("table7a_alarm_performance", pd.DataFrame(alarm_rows),
           "Two operating rules, neither using test labels. The alert-budget rows rank "
           "districts within each week and alert the top k% - deployable under fixed "
           "field capacity, and unaffected by base-rate shift. The sensitivity-target "
           f"rows fit a threshold on {SEL_VALID} and freeze it; they are reported to show "
           "that such a threshold does NOT transfer, because the alarm base rate falls "
           "from 0.372 in 2023 to 0.143 in the test years")
LEAD = save_table("table7b_alarm_lead_time", pd.DataFrame(lead_rows),
                  f"Lead time at h={H_ALARM} under EVERY operating rule tested, not at a "
                  "chosen one. The spread across rules is the result: the median ranges "
                  "from a few weeks to most of a season depending only on how often the "
                  "rule is allowed to fire, so this design does not identify lead time "
                  "and no single figure from this table should be quoted as the warning "
                  "the system provides")
save_table("table7c_divisional_alarm", pd.DataFrame(div_rows),
           f"Alarm discrimination by division, h={H_ALARM}. ROC and PR are threshold-free "
           "and therefore the only alarm numbers that do not depend on the operating rule")
if len(LEAD):
    log.info("[audit] lead time by operating rule (median wk): %s - range %.0f-%.0f, "
             "so lead time is not identifiable in this design",
             dict(zip(LEAD.operating_rule, LEAD.median_lead_weeks)),
             LEAD.median_lead_weeks.min(), LEAD.median_lead_weeks.max())

In [ ]:
# =============================================================================
# CELL 10b — What the alarm gets wrong, and whether its probabilities mean anything
# =============================================================================
# Two gaps the audit found. (a) Twenty-five tables and none of them said what the
# model gets wrong. (b) A paper arguing that calibration is under-examined never
# checked the calibration of its own alarm probabilities.
_h = H_ALARM
_yc = f"target_alarm_{_h}w"
_sub = MD[MD[_yc].notna()]
_tr = _sub[_sub.year <= ALARM_TRAIN_MAX]
_te = _sub[_sub.year.isin(TEST_YEARS)].copy()
_te["p"] = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                    .fit(_tr[FD_FULL], _tr[_yc].astype(int))
                    .predict_proba(_te[FD_FULL])[:, 1] for sd in SEEDS], axis=0)
_te["_y"] = _te[_yc].astype(int)
_te["_tert"] = _te["unit"].map({u: _TERT_NAME[g] for u, g in BURDEN_G.items()})

# --- (a) probability calibration -------------------------------------------------
_brier = float(np.mean((_te["p"] - _te["_y"]) ** 2))
_base = float(_te["_y"].mean())
_bss = 1 - _brier / (_base * (1 - _base)) if 0 < _base < 1 else np.nan
cal_rows = [{"bin": f"{lo:.1f}-{hi:.1f}",
             "n": int(m.sum()),
             "mean_predicted": round(float(_te.loc[m, "p"].mean()), 4) if m.sum() else np.nan,
             "observed_frequency": round(float(_te.loc[m, "_y"].mean()), 4) if m.sum() else np.nan}
            for lo, hi in zip(np.arange(0, 1, 0.1), np.arange(0.1, 1.01, 0.1))
            for m in [(_te["p"] >= lo) & (_te["p"] < hi if hi < 1 else _te["p"] <= 1)]]
CALP = pd.DataFrame([r for r in cal_rows if r["n"] > 0])
if len(CALP):
    _ece = float((CALP["n"] / CALP["n"].sum() *
                  (CALP["mean_predicted"] - CALP["observed_frequency"]).abs()).sum())
else:
    _ece = np.nan
CALP.attrs["brier"] = _brier
save_table("table15_alarm_probability_calibration", CALP,
           f"Reliability of the h={_h} alarm probabilities on the test years. "
           f"Brier {_brier:.4f}, Brier skill score vs the base rate {_bss:.4f}, "
           f"expected calibration error {_ece:.4f}. The paper argues interval "
           "calibration is neglected; this holds the classification arm to the "
           "same standard")
print(CALP.to_string(index=False))
log.info("alarm probabilities: Brier %.4f, BSS %.4f, ECE %.4f", _brier, _bss, _ece)

# --- (b) what it gets wrong, at the frozen threshold ------------------------------
# Errors are characterised at the rule a health unit would actually run: the weekly
# top-10% alert budget. The frozen sensitivity threshold fires on nearly everything in
# the test years, so its "errors" would describe a broken operating point rather than
# the model.
_wr = _te.groupby("week_start")["p"].rank(ascending=False, method="first")
_wn = _te.groupby("week_start")["p"].transform("size")
_te["_pred"] = (_wr <= np.maximum(1, np.round(0.10 * _wn))).astype(int)

err_rows = []
if True:
    for key, grp in [("burden tertile", "_tert"), ("division", "block")]:
        for gname, g in _te.groupby(grp, observed=True):
            tp = int(((g._pred == 1) & (g._y == 1)).sum()); fn = int(((g._pred == 0) & (g._y == 1)).sum())
            fp = int(((g._pred == 1) & (g._y == 0)).sum()); tn = int(((g._pred == 0) & (g._y == 0)).sum())
            err_rows.append({
                "stratum_type": key, "stratum": str(gname), "n_unit_weeks": len(g),
                "outbreak_weeks": int(g._y.sum()),
                "sensitivity": round(tp / (tp + fn), 3) if tp + fn else np.nan,
                "missed_outbreak_weeks": fn,
                "false_alarms": fp,
                "false_alarm_rate": round(fp / (fp + tn), 4) if fp + tn else np.nan,
                "median_cases_when_missed": round(float(g.loc[(g._pred == 0) & (g._y == 1),
                                                              "cases_lag0"].median()), 1)
                if fn else np.nan,
                "median_cases_when_false_alarm": round(float(g.loc[(g._pred == 1) & (g._y == 0),
                                                                   "cases_lag0"].median()), 1)
                if fp else np.nan})
ERR = save_table("table14_error_analysis", pd.DataFrame(err_rows),
                 f"Where the h={_h} alarm fails under the weekly top-10% alert budget, by "
                 "burden tertile and by division. A system that misses outbreaks only in "
                 "low-burden districts is a different proposition from one that misses "
                 "them everywhere")
print(ERR.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 11 — Covariate-group ablation. Do satellite / trends / mobility earn a place?
# =============================================================================
ABLATION = {
    "case history + season":  G_AR + G_SEASON,
    "+ climate":              G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA,
    "+ satellite":            G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC,
    "+ search trends":        G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC + G_TRENDS,
    # G_SURV used to be bundled in with mobility, so its contribution was never
    # visible. Split into two rungs: the paper can now say what each is worth.
    "+ mobility & spillover": G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC + G_TRENDS + G_MOBILITY,
    "+ surveillance":         G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC + G_TRENDS + G_MOBILITY + G_SURV,
}
rows = []
for h in [1, 2, 4]:
    base = None
    for name, cols in ABLATION.items():
        F = select_features(MD, [cols])
        P = rolling_origin(MD, F, h)
        mae = mean_absolute_error(P.y, P.anchored)
        pers = mean_absolute_error(P.y, P.persistence)
        base = mae if base is None else base
        rows.append({"horizon_weeks": h, "feature_set": name, "n_features": len(F),
                     "MAE_anchored": round(mae, 3),
                     "skill_vs_persistence_pct": round(100 * (1 - mae / pers), 2),
                     "delta_MAE_vs_case_history": round(mae - base, 3)})
abl = save_table("table8_covariate_ablation", pd.DataFrame(rows),
                 "Marginal value of each covariate group, anchored-growth model")

# Leave-one-climate-variable-out. Al Mobin (2024, Sci Rep) reports that relative
# humidity is redundant for Bangladeshi dengue prediction; that was concluded from a
# wrapper search on national monthly data with the scaler fitted before the split, so
# it is worth testing directly at district-week resolution under prospective evaluation.
FULL_CLIM = G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA
rows = []
for h in [1, 2]:
    base_F = select_features(MD, [FULL_CLIM])
    P0 = rolling_origin(MD, base_F, h, arima_tag="district")
    pers = mean_absolute_error(P0.y, P0.persistence)
    mae0 = mean_absolute_error(P0.y, P0.anchored)
    # A leave-one-out delta only means something if it is larger than the noise the
    # same model shows across seeds. Reference: sd of the seed members' own MAE.
    sc = [f"__seed{sd}__anchored_{GROWTH_OBJ_NAME}" for sd in SEEDS]
    sc = [c for c in sc if c in P0.columns]
    seed_sd = (float(np.std([mean_absolute_error(P0.y, P0[c]) for c in sc], ddof=1))
               if len(sc) > 1 else np.nan)
    rows.append({"horizon_weeks": h, "variable_removed": "(none)", "n_features": len(base_F),
                 "MAE_anchored": round(mae0, 3), "delta_MAE_vs_full": 0.0,
                 "seed_noise_MAE_sd": round(seed_sd, 3), "exceeds_seed_noise": False,
                 "skill_vs_persistence_pct": round(100 * (1 - mae0 / pers), 2)})
    for v in CLIM_BASE:
        F = select_features(MD, [[c for c in FULL_CLIM if not c.startswith(v)]])
        P = rolling_origin(MD, F, h, arima_tag="district")
        mae = mean_absolute_error(P.y, P.anchored)
        d = mae - mae0
        rows.append({"horizon_weeks": h, "variable_removed": v, "n_features": len(F),
                     "MAE_anchored": round(mae, 3), "delta_MAE_vs_full": round(d, 3),
                     "seed_noise_MAE_sd": round(seed_sd, 3),
                     "exceeds_seed_noise": bool(abs(d) > seed_sd) if np.isfinite(seed_sd) else None,
                     "skill_vs_persistence_pct": round(100 * (1 - mae / pers), 2)})
loo = save_table("table8b_climate_leave_one_out", pd.DataFrame(rows),
                 "Leave-one-climate-variable-out. A positive delta means removing the "
                 "variable hurt. Compare every delta against seed_noise_MAE_sd before "
                 "reading anything into it: at this resolution none of them clears it, so "
                 "individual climate contributions are not separately identifiable")
print(loo.to_string(index=False))
if "exceeds_seed_noise" in loo:
    log.info("climate leave-one-out: %d of %d variables move MAE by more than the "
             "seed-to-seed noise of the same model - individual climate contributions "
             "are not separately identifiable at this resolution",
             int(loo.exceeds_seed_noise.fillna(False).sum()), int((loo.variable_removed != "(none)").sum()))
print(abl[abl.horizon_weeks == 2].to_string(index=False))

In [ ]:
# =============================================================================
# CELL 12 — Resolution ablation. Same code path, shared features, both panels.
# =============================================================================
def resolution_summary(M, F, tag):
    out = []
    for h in HORIZONS:
        P = rolling_origin(M, F, h)
        if not len(P):
            continue
        pers = mean_absolute_error(P.y, P.persistence)
        for m in ["level_Tweedie", "level_L2", "anchored_L1", "anchored_L2"]:
            out.append({"resolution": tag, "n_units": M.unit.nunique(), "horizon_weeks": h,
                        "model": LABELS[m], "key": m,
                        "MAE": round(mean_absolute_error(P.y, P[m]), 3),
                        "skill_vs_persistence_pct": round(100 * (1 - mean_absolute_error(P.y, P[m]) / pers), 2)})
    return pd.DataFrame(out)

res = resolution_summary(MD, FD_SHR, "district (64 units)")
if MV is not None:
    res = pd.concat([res, resolution_summary(MV, FV_SHR, "divisional (8 units)")], ignore_index=True)
res = save_table("table9_resolution_ablation", res,
                 f"Skill by spatial resolution, SHARED feature set only "
                 f"({len(FD_SHR)} features, identical at both resolutions). These skill "
                 f"numbers are lower than table2's ({len(FD_FULL)} features) by "
                 "construction and the two must not be compared row to row")

# Contribution 2 is a difference-in-differences claim: the anchored-minus-level skill
# gap is LARGER at coarse resolution. That needs a test, not two columns side by side.
def resolution_did(h=2, a="anchored_L1", b="level_Tweedie", n_boot=2000, seed=SEED,
                   fd=None, fv=None):
    if MV is None:
        return None
    Pd, Pv = rolling_origin(MD, fd or FD_SHR, h), rolling_origin(MV, fv or FV_SHR, h)
    if not len(Pd) or not len(Pv):
        return None
    rng = np.random.default_rng(seed)

    def gap(P, units):
        sub = P[P.unit.isin(units)]
        if not len(sub):
            return np.nan
        base = mean_absolute_error(sub.y, sub.persistence)
        if base == 0:
            return np.nan
        return (100 * (1 - mean_absolute_error(sub.y, sub[a]) / base)
                - 100 * (1 - mean_absolute_error(sub.y, sub[b]) / base))

    ud, uv = Pd.unit.unique(), Pv.unit.unique()
    obs = gap(Pv, uv) - gap(Pd, ud)
    draws = np.array([gap(Pv, rng.choice(uv, len(uv), replace=True))
                      - gap(Pd, rng.choice(ud, len(ud), replace=True)) for _ in range(n_boot)])
    draws = draws[np.isfinite(draws)]
    lo, hi = np.percentile(draws, [2.5, 97.5])
    pv = 2 * min((draws <= 0).mean(), (draws >= 0).mean())
    return {"horizon_weeks": h, "gap_divisional_pp": round(gap(Pv, uv), 2),
            "gap_district_pp": round(gap(Pd, ud), 2),
            "difference_in_differences_pp": round(float(obs), 2),
            "boot_lo": round(float(lo), 2), "boot_hi": round(float(hi), 2),
            "boot_p": round(float(pv), 4),
            "verdict": ("gap is larger at coarse resolution" if lo > 0 else
                        "gap is larger at fine resolution" if hi < 0 else
                        "no significant resolution effect")}

did = [d for d in (resolution_did(h) for h in HORIZONS) if d]
if did:
    D = pd.DataFrame(did)
    _rej, _adj = bh_fdr(D["boot_p"].values, q=ALPHA)
    D["boot_p_BH"] = np.round(_adj, 4)
    save_table("table9b_resolution_significance", D,
               "Difference-in-differences test of the resolution-dependence claim")
    print(D.to_string(index=False))
print(res.pivot_table(index=["horizon_weeks"], columns=["resolution", "key"],
                      values="skill_vs_persistence_pct").round(1).to_string())

In [ ]:
# =============================================================================
# CELL 12b — Sensitivity analyses for the two data-quality defects found in the audit
# =============================================================================
sens_rows = []

# --- A. Zeros that are probably MISSING REPORTS rather than true zeros -------
# table1b flags weeks with cases = 0 while patients are still admitted. If those
# rows drive the headline, the zero-inflation story is an artefact of missingness.
# Key on integer nanoseconds, not on datetime objects. Under numpy 2.0.2 a
# np.datetime64 and a pd.Timestamp for the same instant hash DIFFERENTLY, so a
# (unit, week) tuple built from .values never matched a key built from a DataFrame,
# _is_gap returned all-False, and this whole sensitivity silently dropped zero rows
# while reporting "no change". It matched under numpy 2.5 locally and not on Kaggle,
# which is the worst kind of bug: a robustness check that passes by doing nothing.
def _wk_ns(weeks):
    # astype("datetime64[ns]") is not decoration. pd.to_datetime PRESERVES the input
    # resolution, so the same instant becomes seconds from a datetime64[D] column,
    # microseconds from a Timestamp Series and nanoseconds from datetime64[ns] - three
    # different integers for one date. Pinning the unit is what makes the key comparable.
    return (pd.to_datetime(pd.Series(np.asarray(weeks)).reset_index(drop=True))
            .astype("datetime64[ns]").astype("int64"))


assert "currently_admitted" in PANEL_D.columns, (
    "the report-gap sensitivity needs currently_admitted; without it the check is "
    "vacuous and must not be reported as passing")
_gap_rows = PANEL_D.loc[(PANEL_D.cases == 0) & (PANEL_D["currently_admitted"] > 0),
                        ["unit", "week_start"]]
gap_key = set(zip(_gap_rows["unit"].astype(str), _wk_ns(_gap_rows["week_start"])))
log.info("report-gap weeks flagged: %d", len(gap_key))
assert gap_key, "no report-gap weeks found - expected 791; the filter is not working"


def _is_gap(units, weeks):
    return np.fromiter(((u, w) in gap_key
                        for u, w in zip(np.asarray(units).astype(str), _wk_ns(weeks))),
                       dtype=bool, count=len(units))

for h in [1, 2]:
    P = PRED[h].copy()
    tgt_week = P["week_start"] + pd.to_timedelta(7 * h, unit="D")
    drop = _is_gap(P.unit.values, P.week_start.values) | _is_gap(P.unit.values, tgt_week.values)
    keep = P[~drop]
    # If gap weeks fall inside this prediction window, removing them must remove
    # something. Zero means the match failed, not that the data is clean.
    _in_window = _is_gap(P.unit.values, P.week_start.values).sum()
    assert not (_in_window > 0 and len(keep) == len(P)), (
        f"h={h}: {_in_window} report-gap rows are present but none were removed - "
        "the sensitivity is vacuous and cannot be reported as showing no change")
    for tag, sub in [("all rows", P), ("report-gap rows removed", keep)]:
        base = mean_absolute_error(sub.y, sub.persistence)
        sens_rows.append({
            "sensitivity": "A · missing-report zeros", "horizon_weeks": h, "variant": tag,
            "n_rows": len(sub), "rows_dropped_pct": round(100 * (1 - len(sub) / len(P)), 2),
            "metric": "skill vs persistence (%)",
            "anchored": round(100 * (1 - mean_absolute_error(sub.y, sub.anchored) / base), 2),
            "level_Tweedie": round(100 * (1 - mean_absolute_error(sub.y, sub.level_Tweedie) / base), 2)})
    I = INTERVALS[(h, "split")].copy()
    tw = I["week_start"] + pd.to_timedelta(7 * h, unit="D")
    dr = _is_gap(I.unit.values, I.week_start.values) | _is_gap(I.unit.values, tw.values)
    for tag, sub in [("all rows", I), ("report-gap rows removed", I[~dr])]:
        sens_rows.append({
            "sensitivity": "A · missing-report zeros (coverage)", "horizon_weeks": h, "variant": tag,
            "n_rows": len(sub), "rows_dropped_pct": round(100 * (1 - len(sub) / len(I)), 2),
            "metric": "interval coverage (fraction)",
            "anchored": round(float(((sub.y >= sub.lo) & (sub.y <= sub.hi)).mean()), 4),
            "level_Tweedie": np.nan})

# --- B. Rainfall is CHIRPS in the district panel, NASA POWER in the divisional one
# table1c flags this (r = 0.86). If the resolution result survives dropping rain
# entirely, the residual source mismatch is not what produces it.
RAIN_COLS = [c for c in FD_SHR if c.startswith("rain")]
fd_norain = [c for c in FD_SHR if c not in RAIN_COLS]
fv_norain = [c for c in FV_SHR if not c.startswith("rain")]
log.info("rain columns removed for sensitivity B: %s", RAIN_COLS)
for h in HORIZONS:
    for tag, kw in [("with rain", {}), ("rain removed", dict(fd=fd_norain, fv=fv_norain))]:
        d_ = resolution_did(h, n_boot=800, **kw)
        if d_:
            sens_rows.append({"sensitivity": "B · rain source mismatch", "horizon_weeks": h,
                              "variant": tag, "n_rows": np.nan, "rows_dropped_pct": np.nan,
                              "metric": "district-vs-divisional DiD (pp)",
                              "anchored": d_["difference_in_differences_pp"],
                              "level_Tweedie": np.nan,
                              "boot_lo": d_["boot_lo"], "boot_hi": d_["boot_hi"],
                              "boot_p": d_["boot_p"], "verdict": d_["verdict"]})

# Three different quantities live in the `anchored` column - a skill percentage, a
# coverage fraction and a difference-in-differences in percentage points. They are not
# comparable to each other, so `metric` states the units of every row and nothing is
# readable as a skill score by accident.
SENS = save_table("table11_data_quality_sensitivity", pd.DataFrame(sens_rows),
                  "Do the two audit defects change the conclusions? "
                  "A: missing-report zeros. B: CHIRPS-vs-POWER rainfall mismatch. "
                  "Read `anchored`/`level_Tweedie` in the units given by `metric`")
print(SENS.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 12e — Would the results survive real reporting latency?
# =============================================================================
# The audit's hidden assumption: satellite composites, search trends and same-week
# surveillance are used at week t as though a forecaster standing at t already had
# them. MODIS and VIIRS carry days-to-weeks processing latency, Google Trends is
# revised, and DGHS counts are backfilled. No vintaged archive exists to settle this
# empirically, but the assumption CAN be stress-tested: delay every latent covariate
# by one and two extra weeks and re-run. If the headline results hold, the same-week
# assumption is not doing the work.
LATENT = [c for c in (G_SAT + G_TRENDS + G_SURV) if c in MD.columns]
log.info("latency stress test delays %d covariates: %s", len(LATENT), LATENT[:6])

# The delays tested run out to six weeks because the Cox's Bazar cross-check
# (table18) puts the empirical onset-to-report lag at five weeks, not one or two. A
# stress test gentler than the lag the data actually shows would prove nothing.
lat_rows = []
for _delay in [0, 1, 2, 4, 6]:
    Mx = MD.copy()
    if _delay:
        Mx = Mx.sort_values(["unit", "week_start"])
        Mx[LATENT] = Mx.groupby("unit")[LATENT].shift(_delay)
        Mx[LATENT] = Mx.groupby("unit")[LATENT].transform(lambda c: c.ffill())
    for _h in [1, 2]:
        Px = rolling_origin(Mx, FD_FULL, _h, arima_tag=None)
        _b = mean_absolute_error(Px.y, Px.persistence)
        _sk = 100 * (1 - mean_absolute_error(Px.y, Px.anchored) / _b)
        Ix = interval_run(Mx, FD_FULL, _h, "mondrian_adaptive", groups=BURDEN_G)
        Ix["_g"] = Ix["unit"].map(BURDEN_G)
        _cov = float(((Ix.y >= Ix.lo) & (Ix.y <= Ix.hi)).mean())
        _agg = (Ix.assign(_c=((Ix.y >= Ix.lo) & (Ix.y <= Ix.hi)).astype(int))
                  .groupby("_g")["_c"].agg(["sum", "count"]))
        _gap = np.nan
        if {0, 2} <= set(_agg.index):
            _gap = (_agg.loc[0, "sum"] / _agg.loc[0, "count"]
                    - _agg.loc[2, "sum"] / _agg.loc[2, "count"])
        lat_rows.append({"extra_delay_weeks": _delay, "horizon_weeks": _h,
                         "anchored_skill_pct": round(float(_sk), 2),
                         "coverage_group_conditional": round(_cov, 4),
                         "burden_gap": round(float(_gap), 4) if np.isfinite(_gap) else np.nan})

LAT = save_table("table16_reporting_latency", pd.DataFrame(lat_rows),
                 "Every satellite, search-trend and same-week surveillance covariate "
                 "delayed by 1 and 2 extra weeks, simulating publication latency. The "
                 "study cannot vintage its inputs, so this is the closest available test "
                 "of whether the same-week availability assumption carries the results")
print(LAT.to_string(index=False))
if len(LAT):
    _d0 = LAT[LAT.extra_delay_weeks == 0].set_index("horizon_weeks")
    _d2 = LAT[LAT.extra_delay_weeks == 2].set_index("horizon_weeks")
    log.info("[audit] two-week latency costs %s skill points; coverage moves %s",
             list((_d0.anchored_skill_pct - _d2.anchored_skill_pct).round(2)),
             list((_d2.coverage_group_conditional - _d0.coverage_group_conditional).round(4)))

In [ ]:
# =============================================================================
# CELL 12f — Does the optimism gap depend on how an outbreak is defined at all?
# =============================================================================
# Cell 9b swept the quantile and the floor, but both are the SAME kind of definition:
# a per-district level threshold. If the gap only exists for level thresholds it is a
# property of that construct rather than of the validation design. Two structurally
# different definitions are added here: a WHO-style endemic channel (the district's own
# mean + 2 SD for that epidemiological week, estimated on training years only) and a
# growth-based definition (a sustained doubling over four weeks), which does not use a
# level threshold at all.
def _label_endemic_channel(M, k=2.0):
    base = M[(M.year >= ALARM_BASE_MIN) & (M.year <= ALARM_TRAIN_MAX)]
    st = base.groupby(["unit", "epi_week"])["cases"].agg(["mean", "std"]).reset_index()
    st["chan"] = np.maximum(st["mean"] + k * st["std"].fillna(0.0), ALARM_MIN_CASES)
    ch = M.merge(st[["unit", "epi_week", "chan"]], on=["unit", "epi_week"], how="left")
    return np.maximum(ch["chan"].fillna(ALARM_MIN_CASES).values, ALARM_MIN_CASES)


def _label_growth(M, h, mult=2.0):
    lead = M[f"target_lead_{h}w"]
    base = M["cases_lag0"].astype(float)
    lab = ((lead >= mult * base) & (lead >= ALARM_MIN_CASES)).astype(float)
    return lab.where(lead.notna())


defn_rows = []
for _name in ["per-district quantile (reported)", "endemic channel (mean + 2SD)",
              "growth: doubling over the horizon"]:
    Md = MD.copy()
    _lead = Md[f"target_lead_{H_ALARM}w"]
    if _name.startswith("per-district"):
        _lab = Md[f"target_alarm_{H_ALARM}w"]
    elif _name.startswith("endemic"):
        _chan = _label_endemic_channel(Md)
        _lab = (_lead >= _chan).astype(float).where(_lead.notna())
    else:
        _lab = _label_growth(Md, H_ALARM)
    Ad = Md.loc[_lab.notna()].copy()
    Ad["_y"] = _lab.loc[_lab.notna()].astype(int).values
    if Ad["_y"].nunique() < 2 or Ad["_y"].sum() < 30:
        continue

    def _fd(tr, te):
        pr = np.mean([lgb.LGBMClassifier(**{**LGB_CLF, "random_state": sd})
                      .fit(tr[FD_FULL], tr["_y"]).predict_proba(te[FD_FULL])[:, 1]
                      for sd in SEEDS], axis=0)
        return pr, te["_y"].values

    def _pd_(pairs):
        ps, ys = zip(*pairs)
        pp, yy = np.concatenate(ps), np.concatenate(ys)
        return float(roc_auc_score(yy, pp)), float(average_precision_score(yy, pp))

    _bl = sorted(Ad["block"].unique())
    _t1, _e1 = train_test_split(Ad, test_size=0.20, random_state=SEED, stratify=Ad["_y"])
    d1 = _pd_([_fd(_t1, _e1)])
    d4 = _pd_([_fd(Ad[(Ad.year < ty) & (Ad.block != b)], Ad[(Ad.year == ty) & (Ad.block == b)])
               for ty in TEST_YEARS for b in _bl
               if Ad[(Ad.year == ty) & (Ad.block == b)]["_y"].nunique() > 1])
    defn_rows.append({"outbreak_definition": _name, "base_rate": round(float(Ad["_y"].mean()), 4),
                      "n_rows": len(Ad), "C1_ROC": round(d1[0], 4), "C4_ROC": round(d4[0], 4),
                      "gap_ROC": round(d1[0] - d4[0], 4),
                      "C1_PR": round(d1[1], 4), "C4_PR": round(d4[1], 4),
                      "gap_PR": round(d1[1] - d4[1], 4)})

DEFN = save_table("table17_outbreak_definition_sensitivity", pd.DataFrame(defn_rows),
                  "The optimism gap under three structurally different outbreak "
                  "definitions - a level threshold, an endemic channel, and a "
                  "growth-rate rule that uses no level threshold. A gap that survives "
                  "all three is a property of the validation design, not of the label")
print(DEFN.to_string(index=False))
if len(DEFN):
    log.info("[audit] optimism gap by outbreak definition: ROC %s, PR %s",
             list(DEFN.gap_ROC), list(DEFN.gap_PR))

In [ ]:
# =============================================================================
# CELL 12g — Independent within-country validation: Cox's Bazar symptom onset
# =============================================================================
# This study is scoped to Bangladesh, so its external validity is validity across
# unseen districts, seasons and DATA SOURCES within Bangladesh - not transfer to
# another country. The deposit holds 35,581 individually recorded Cox's Bazar patients
# with symptom-onset dates, collected through a different system from the DGHS
# aggregate returns the model is trained on. That gives two things nothing else here
# can: an independent reconstruction of one district's epidemic curve, and a direct
# estimate of the reporting delay, because onset precedes reporting by construction.
# CSV first. The workbook is the original deposit format, but Excel reading needs
# openpyxl to be installed, parses dates through a separate code path whose
# resolution has already caused one silent failure in this pipeline, and ships 12 MB
# of binary nobody can diff. The CSV is a verbatim round-trip - same 35,581 rows,
# same onset dates, same weekly aggregation - and keeps every input in one format.
CB_PATH = (_find("coxsbazar_dengue_2021_2024.csv")
           or _find("coxsbazar_dengue_2021_2024.xlsx"))
cb_rows = []
if CB_PATH:
    try:
        _cb = (pd.read_csv(CB_PATH, low_memory=False) if CB_PATH.lower().endswith(".csv")
               else pd.read_excel(CB_PATH))
        _onset = pd.to_datetime(_cb["Symptom Onset"], errors="coerce")
        _cbw = (_onset.dropna().dt.to_period("W").dt.start_time
                .value_counts().sort_index().rename("onset_cases"))
        _cbw.index.name = "week_start"
        _cbw = _cbw.reset_index()

        _pan = (PANEL_D[PANEL_D.unit.str.contains("Cox", case=False, na=False)]
                [["week_start", "cases"]].rename(columns={"cases": "reported_cases"}))
        _pan["week_start"] = pd.to_datetime(_pan["week_start"])
        _cbw["week_start"] = pd.to_datetime(_cbw["week_start"])
        # Align both to Monday-start weeks so the join is not defeated by convention.
        for _df in (_cbw, _pan):
            _df["week_start"] = _df["week_start"] - pd.to_timedelta(
                _df["week_start"].dt.weekday, unit="D")
        _cbw = _cbw.groupby("week_start", as_index=False)["onset_cases"].sum()
        _pan = _pan.groupby("week_start", as_index=False)["reported_cases"].sum()
        _mg = _pan.merge(_cbw, on="week_start", how="inner").sort_values("week_start")

        if len(_mg) >= 30:
            # Cross-correlate to find the delay at which the two series agree best.
            # A positive best lag means reported counts follow onset - the reporting
            # lag the audit flagged as an untested assumption.
            for _lag in range(-2, 7):
                a = _mg["onset_cases"].values
                b = _mg["reported_cases"].shift(-_lag).values
                m = ~np.isnan(b)
                if m.sum() < 25:
                    continue
                r = float(np.corrcoef(a[m], b[m])[0, 1])
                rs = float(stats.spearmanr(a[m], b[m])[0])
                cb_rows.append({"lag_weeks_reported_after_onset": _lag,
                                "n_weeks": int(m.sum()),
                                "pearson_r": round(r, 4), "spearman_rho": round(rs, 4)})
            log.info("Cox's Bazar cross-check: %d overlapping weeks, %d onset cases vs "
                     "%d reported", len(_mg), int(_mg.onset_cases.sum()),
                     int(_mg.reported_cases.sum()))
    except Exception as e:
        log.warning("Cox's Bazar cross-check unavailable: %s", e)

CB = pd.DataFrame(cb_rows)
if not len(CB):
    # An empty table here means the check did not RUN, not that it ran and found
    # nothing. On Kaggle that happens when the Cox's Bazar workbook is not attached
    # to the notebook's dataset, and a blank CSV would otherwise be mistaken for a
    # completed validation.
    log.warning("[gate] Cox's Bazar independent validation DID NOT RUN%s. Attach "
                "coxsbazar_dengue_2021_2024.xlsx to the input dataset; without it "
                "table18 is empty and the paper has no independent within-country "
                "check of the surveillance series.",
                " (file not found)" if not CB_PATH else " (file found but unreadable)")
if len(CB):
    _best = CB.loc[CB.pearson_r.idxmax()]
    CB["best_lag"] = CB.lag_weeks_reported_after_onset == _best.lag_weeks_reported_after_onset
    log.info("[audit] Cox's Bazar: independent onset series correlates r=%.3f with the "
             "DGHS reported series at a %d-week lag (r=%.3f at lag 0)",
             _best.pearson_r, int(_best.lag_weeks_reported_after_onset),
             float(CB[CB.lag_weeks_reported_after_onset == 0].pearson_r.iloc[0])
             if (CB.lag_weeks_reported_after_onset == 0).any() else np.nan)
save_table("table18_coxsbazar_independent_check", CB,
           "One district's epidemic curve rebuilt from 35,581 individual symptom-onset "
           "records and compared with the DGHS aggregate series the models are trained "
           "on. The lag at which correlation peaks is an empirical estimate of the "
           "reporting delay, which no other source in this study can provide")
print(CB.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 12h — Generalization across seasons and districts, within Bangladesh
# =============================================================================
# Prospective evaluation uses two test seasons, which is what the 2020-21 surveillance
# gap allows. Season-to-season VARIANCE is a different question and more of the panel
# can speak to it: hold out each season in turn. Only the rows marked prospective are
# forecasts; the others train on data after the held-out season and are reported as
# generalization probes, not as forecasting skill. Conflating the two would be exactly
# the overstatement this study is trying to avoid.
gen_rows = []
for _season in sorted(MD.year.unique()):
    _tr = MD[(MD.year != _season) & MD["target_lead_2w"].notna()]
    _te = MD[(MD.year == _season) & MD["target_lead_2w"].notna()]
    if len(_te) < 200 or len(_tr) < 500:
        continue
    _pred = np.mean([lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd},
                                       **GROWTH_OBJECTIVES[GROWTH_OBJ_NAME])
                     .fit(_tr[FD_FULL], _tr["target_growth_2w"]).predict(_te[FD_FULL])
                     for sd in SEEDS], axis=0)
    _yhat = np.clip((_te["cases_lag0"].values + 1.0) * np.exp(_pred) - 1, 0, None)
    _y = _te["target_lead_2w"].values
    _b = mean_absolute_error(_y, _te["cases_lag0"].values)
    gen_rows.append({
        "held_out_season": int(_season), "n_unit_weeks": len(_te),
        "cases_in_season": int(_te["cases"].sum()),
        "skill_vs_persistence_pct": round(100 * (1 - mean_absolute_error(_y, _yhat) / _b), 2),
        "evaluation": "prospective" if _season in TEST_YEARS else
                      "retrospective probe (trains on later seasons)"})

# Spatial generalization: how much does prospective skill vary BY DISTRICT?
P2 = PRED[2]
_per_unit = []
for u, g in P2.groupby("unit"):
    _bb = mean_absolute_error(g.y, g.persistence)
    if _bb > 0:
        _per_unit.append(100 * (1 - mean_absolute_error(g.y, g.anchored) / _bb))
_pu = np.asarray(_per_unit, float)
GEN = pd.DataFrame(gen_rows)
save_table("table19_generalization_within_bangladesh", GEN,
           "Skill with each season held out. Two seasons are prospective; the rest are "
           "retrospective probes of season-to-season variance and are labelled as such. "
           f"Across the 64 districts, prospective h=2 skill has median {np.median(_pu):.1f}%, "
           f"IQR {np.percentile(_pu, 25):.1f} to {np.percentile(_pu, 75):.1f}%, and is "
           f"positive in {int((_pu > 0).sum())} of {len(_pu)} districts")
print(GEN.to_string(index=False))
log.info("[audit] per-district prospective skill: median %.1f%%, IQR %.1f-%.1f, "
         "positive in %d of %d districts", np.median(_pu),
         np.percentile(_pu, 25), np.percentile(_pu, 75), int((_pu > 0).sum()), len(_pu))

In [ ]:
# =============================================================================
# CELL 12i — Why does skill vanish on the 2023 season?
# =============================================================================
# table19 shows 0.2% skill when 2023 is held out, against 17-40% for every other
# season. 2023 carried 321,593 cases against roughly 100,000 in the neighbouring
# years, so two explanations compete. Either the model cannot extrapolate to a
# magnitude regime absent from its training data, or persistence simply becomes hard
# to beat when an epidemic grows this fast - in which case the low skill says
# something about the BASELINE rather than about the model. Absolute errors separate
# them: under the first, model error explodes; under the second, both errors explode
# together and their ratio stays near one.
coll_rows = []
for _season in sorted(MD.year.unique()):
    _tr = MD[(MD.year != _season) & MD["target_lead_2w"].notna()]
    _te = MD[(MD.year == _season) & MD["target_lead_2w"].notna()].copy()
    if len(_te) < 200 or len(_tr) < 500:
        continue
    _pr = np.mean([lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd},
                                     **GROWTH_OBJECTIVES[GROWTH_OBJ_NAME])
                   .fit(_tr[FD_FULL], _tr["target_growth_2w"]).predict(_te[FD_FULL])
                   for sd in SEEDS], axis=0)
    _te["_yhat"] = np.clip((_te["cases_lag0"].values + 1.0) * np.exp(_pr) - 1, 0, None)
    _te["_y"] = _te["target_lead_2w"].values
    _te["_pers"] = _te["cases_lag0"].values
    # Tercile by the size of the district-week, so "does it fail on the big weeks?"
    # is answered directly rather than inferred from a pooled average.
    _te["_tier"] = pd.qcut(_te["_pers"].rank(method="first"), 3,
                           labels=["small weeks", "medium weeks", "large weeks"])
    for _tier, g in _te.groupby("_tier", observed=True):
        _mp = mean_absolute_error(g._y, g._pers)
        _mm = mean_absolute_error(g._y, g._yhat)
        coll_rows.append({
            "held_out_season": int(_season), "week_size_tercile": str(_tier),
            "n": len(g), "median_cases": round(float(g._pers.median()), 1),
            "MAE_persistence": round(float(_mp), 2), "MAE_model": round(float(_mm), 2),
            "skill_pct": round(100 * (1 - _mm / _mp), 2) if _mp > 0 else np.nan,
            "mean_bias_model": round(float((g._yhat - g._y).mean()), 2),
            "evaluation": "prospective" if _season in TEST_YEARS else "retrospective probe"})

COLL = save_table("table20_season_collapse_diagnostic", pd.DataFrame(coll_rows),
                  "Absolute errors, not just skill, for each held-out season split by "
                  "how large the district-week is. Skill is a ratio and hides whether a "
                  "low value means the model failed or the baseline got hard to beat")
print(COLL.to_string(index=False))
if len(COLL):
    _c23 = COLL[COLL.held_out_season == 2023]
    _c24 = COLL[COLL.held_out_season == 2024]
    if len(_c23) and len(_c24):
        log.info("[audit] 2023 held out: MAE persistence %s vs model %s | "
                 "2024: persistence %s vs model %s",
                 list(_c23.MAE_persistence), list(_c23.MAE_model),
                 list(_c24.MAE_persistence), list(_c24.MAE_model))

In [ ]:
# =============================================================================
# CELL 12c — Seed stability. How much would a single-seed run have moved?
# =============================================================================
# The Kaggle and local runs of the previous notebook version disagreed by up to
# 33 skill points on one model and flipped two significance verdicts, because
# every number came from a single LightGBM fit. Headline models are now ensembles
# over SEEDS; this table reports what the individual members did, so the paper can
# quote a spread instead of a number that does not reproduce.
stab_rows = []
for h in HORIZONS:
    P = PRED[h]
    base = mean_absolute_error(P.y, P.persistence)
    for model in ["level_L2", "level_Tweedie", "anchored_L1", "anchored_L2"]:
        cols = [f"__seed{sd}__{model}" for sd in SEEDS if f"__seed{sd}__{model}" in P.columns]
        if not cols:
            continue
        per = [100 * (1 - mean_absolute_error(P.y, P[c]) / base) for c in cols]
        ens = 100 * (1 - mean_absolute_error(P.y, P[model]) / base)
        stab_rows.append({
            "horizon_weeks": h, "model": LABELS.get(model, model), "key": model,
            "ensemble_skill_pct": round(ens, 2),
            "single_seed_min_pct": round(float(np.min(per)), 2),
            "single_seed_max_pct": round(float(np.max(per)), 2),
            "single_seed_sd_pct": round(float(np.std(per, ddof=1)) if len(per) > 1 else 0.0, 2),
            "n_seeds": len(per)})
STAB = save_table("table12_seed_stability", pd.DataFrame(stab_rows),
                  "Ensemble skill against the spread of its individual seeds")
print(STAB.to_string(index=False))
if len(STAB):
    log.info("worst single-seed spread across all models/horizons: %.1f skill points",
             (STAB.single_seed_max_pct - STAB.single_seed_min_pct).max())

In [ ]:
# =============================================================================
# CELL 12d — Properly powered comparison of the two target parameterisations
# =============================================================================
# The audit showed this comparison is unresolvable at three seeds: at h=2 the
# anchored model's seed-to-seed sd was 5.95 skill points against a claimed
# advantage of 2.53. Here each of N_PAIR_SEEDS seeds fits BOTH models on the same
# folds, so seed is a blocking factor and the comparison is paired - far more
# efficient than comparing two independent ensembles. A paired t-test and a
# Wilcoxon signed-rank across seeds then answer the question directly.
N_PAIR_SEEDS = int(os.environ.get("ICEEICT_PAIR_SEEDS", 20))
PAIR_SEEDS = list(range(1000, 1000 + N_PAIR_SEEDS))
log.info("paired seed comparison: %d seeds", N_PAIR_SEEDS)

pair_rows = []
for h in HORIZONS:
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    per_seed = {"anchored": [], "level": []}
    for sd in PAIR_SEEDS:
        ya, yl, yy, pp = [], [], [], []
        for ty in TEST_YEARS:
            tr = MD[(MD.year < ty) & MD[tl].notna()]
            te = MD[(MD.year == ty) & MD[tl].notna()]
            anchor_v = te["cases_lag0"].values + 1.0
            mg = lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd},
                                   **GROWTH_OBJECTIVES[GROWTH_OBJ_NAME]).fit(tr[FD_FULL], tr[tg])
            ya.append(np.clip(anchor_v * np.exp(mg.predict(te[FD_FULL])) - 1, 0, None))
            ml = lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd},
                                   objective="tweedie",
                                   tweedie_variance_power=TWEEDIE_P).fit(tr[FD_FULL], tr[tl])
            yl.append(np.clip(ml.predict(te[FD_FULL]), 0, None))
            yy.append(te[tl].values); pp.append(te["cases_lag0"].values)
        y = np.concatenate(yy); base = mean_absolute_error(y, np.concatenate(pp))
        per_seed["anchored"].append(100 * (1 - mean_absolute_error(y, np.concatenate(ya)) / base))
        per_seed["level"].append(100 * (1 - mean_absolute_error(y, np.concatenate(yl)) / base))
    a = np.array(per_seed["anchored"]); l = np.array(per_seed["level"]); d = a - l
    t_stat, p_t = stats.ttest_rel(a, l)
    try:
        _, p_w = stats.wilcoxon(a, l)
    except Exception:
        p_w = np.nan
    dz = float(d.mean() / d.std(ddof=1)) if d.std(ddof=1) > 0 else np.nan   # Cohen dz
    se = d.std(ddof=1) / np.sqrt(len(d))
    pair_rows.append({
        "horizon_weeks": h, "n_seeds": len(d),
        "anchored_mean_skill": round(float(a.mean()), 2), "anchored_sd": round(float(a.std(ddof=1)), 2),
        "level_mean_skill": round(float(l.mean()), 2), "level_sd": round(float(l.std(ddof=1)), 2),
        "paired_diff": round(float(d.mean()), 3),
        "ci_lo": round(float(d.mean() - 1.96 * se), 3), "ci_hi": round(float(d.mean() + 1.96 * se), 3),
        "cohens_dz": round(dz, 3), "p_paired_t": round(float(p_t), 5),
        "p_wilcoxon": round(float(p_w), 5) if np.isfinite(p_w) else np.nan})

PAIR = pd.DataFrame(pair_rows)
_rej, _adj = bh_fdr(PAIR["p_paired_t"].values, q=ALPHA)
PAIR["p_BH"] = np.round(_adj, 5)
PAIR["verdict"] = np.where(_rej, np.where(PAIR.paired_diff > 0,
                                          "anchored better", "level better"),
                           "no detectable difference")
save_table("table13_paired_seed_comparison", PAIR,
           f"Anchored growth vs Tweedie level across {N_PAIR_SEEDS} paired seeds; "
           "seed is a blocking factor so the comparison is paired, not two ensembles")
print(PAIR[["horizon_weeks", "anchored_mean_skill", "anchored_sd", "level_mean_skill",
            "level_sd", "paired_diff", "ci_lo", "ci_hi", "cohens_dz", "p_BH", "verdict"]]
      .to_string(index=False))
log.info("paired comparison resolves %d of %d horizons at BH q<%.2f",
         int(_rej.sum()), len(PAIR), ALPHA)

In [ ]:
# =============================================================================
# CELL 13 — 2026 forward test. Frozen at end-2025, never refitted.
# =============================================================================
fwd_rows = []
if (MD.year == 2026).any():
    for h in [1, 2, 4]:
        tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
        tr_all = MD[(MD.year <= 2025) & MD[tg].notna()].sort_values("week_start")
        te = MD[(MD.year == 2026) & MD[tl].notna()]
        if len(te) < 20:
            continue
        _cw = CAL_WEEKS_BY_H.get(h, CAL_WEEKS)
        _wk = np.sort(tr_all["week_start"].unique())
        _idx = np.unique(np.linspace(0, len(_wk) - 1, _cw).round().astype(int))
        _isc = tr_all["week_start"].isin(set(_wk[_idx]))
        tr, cal = tr_all[~_isc], tr_all[_isc]
        # Same objective and same seed ensemble as the headline anchored model,
        # so the forward test measures the model the paper actually reports.
        gobj = GROWTH_OBJECTIVES[GROWTH_OBJ_NAME]
        anchor = te["cases_lag0"].values + 1.0
        pred = np.clip(anchor * np.exp(np.mean(
            [lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd}, **gobj)
             .fit(tr[FD_FULL], tr[tg]).predict(te[FD_FULL]) for sd in SEEDS], axis=0)) - 1, 0, None)
        q = {a: _EnsQ([lgb.LGBMRegressor(**{**LGB_REG, "random_state": sd},
                                         objective="quantile", alpha=a).fit(tr[FD_FULL], tr[tg])
                       for sd in SEEDS]) for a in (0.05, 0.95)}
        cl, ch = q[0.05].predict(cal[FD_FULL]), q[0.95].predict(cal[FD_FULL])
        _wf = max(float(np.quantile(ch - cl, 0.05)), 1e-3)
        cw_ = np.maximum(ch - cl, _wf)
        s = np.maximum(cl - cal[tg].values, cal[tg].values - ch) / cw_
        n = len(s); qh = np.quantile(s, min(1.0, np.ceil((n + 1) * NOMINAL) / n))
        tl_, th_ = q[0.05].predict(te[FD_FULL]), q[0.95].predict(te[FD_FULL])
        tw_ = np.maximum(th_ - tl_, _wf)
        lo = np.clip(anchor * np.exp(tl_ - qh * tw_) - 1, 0, None)
        hi = anchor * np.exp(th_ + qh * tw_) - 1
        y = te[tl].values
        fwd_rows.append({"horizon_weeks": h, "n_unit_weeks": len(te),
                         "weeks_covered": int(te.epi_week.nunique()),
                         "MAE_anchored": round(mean_absolute_error(y, pred), 3),
                         "MAE_persistence": round(mean_absolute_error(y, te.cases_lag0), 3),
                         "skill_vs_persistence_pct": round(100 * (1 - mean_absolute_error(y, pred) /
                                                                  mean_absolute_error(y, te.cases_lag0)), 2),
                         "conformal_coverage": round(float(((y >= lo) & (y <= hi)).mean()), 4)})
fwd = save_table("table10_forward_test_2026", pd.DataFrame(fwd_rows),
                 "True forward test: model frozen at end-2025, applied to 2026")
if len(fwd):
    print(fwd.to_string(index=False))
else:
    log.warning("no 2026 rows available — forward test skipped")

In [ ]:
# =============================================================================
# CELL 14 — Publication figures (300 dpi, IEEE single-column, CVD-safe)
# =============================================================================
plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8.5,
    "axes.titlesize": 9, "legend.fontsize": 7.5, "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5, "figure.dpi": 300, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.4,
    "axes.axisbelow": True, "lines.linewidth": 1.4, "lines.markersize": 4,
})
COL1, COL2 = 3.45, 7.16       # IEEE single / double column width, inches

def _save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIGURES, f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig)
    log.info("figure -> %s.{png,pdf}", name)

# --- F1: national weekly series, folds shaded, data gap marked ----------------
nat = PANEL_D.groupby("week_start")["cases"].sum().reset_index()
fig, ax = plt.subplots(figsize=(COL2, 2.3))
ax.plot(nat.week_start, nat.cases, color=PAL["blue"], lw=1.2)
ax.fill_between(nat.week_start, 0, nat.cases, color=PAL["blue"], alpha=0.12)
for ty, c in zip(TEST_YEARS, [PAL["verm"], PAL["green"]]):
    ax.axvspan(pd.Timestamp(f"{ty}-01-01"), pd.Timestamp(f"{ty}-12-31"), color=c, alpha=0.10)
    ax.text(pd.Timestamp(f"{ty}-07-01"), nat.cases.max() * 0.94, f"test {ty}",
            ha="center", fontsize=7, color=c)
ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2021-12-31"), color="0.85", alpha=0.6)
ax.text(pd.Timestamp("2021-01-01"), nat.cases.max() * 0.55, "no district-wise\nreporting",
        ha="center", fontsize=7, color="0.35")
ax.set_ylabel("Reported cases per week"); ax.set_xlabel("")
ax.set_title("National weekly dengue incidence, 64 districts aggregated", loc="left")
_save(fig, "fig1_national_series_folds")

# --- F2: skill vs horizon, one axis, direct labels ---------------------------
fig, ax = plt.subplots(figsize=(COL2 * 0.62, 2.5))
series = [("level_L2", PAL["purple"], "o", "Level (L2)"),
          ("level_Tweedie", PAL["verm"], "s", "Level (Tweedie)"),
          ("anchored_L1", PAL["green"], "^", "Anchored growth (L1)")]
# Direct labels at the line ends; no legend box sitting over the data.
for key, c, mk, lab in series:
    v = [shootout[(shootout.horizon_weeks == h) & (shootout.key == key)]
         .skill_vs_persistence_pct.iloc[0] for h in HORIZONS]
    ax.plot(HORIZONS, v, color=c, marker=mk)
    ax.annotate(f"{lab}  {v[-1]:+.0f}%", (HORIZONS[-1], v[-1]), textcoords="offset points",
                xytext=(7, 0), fontsize=7, color=c, va="center", annotation_clip=False)
ax.axhline(0, color=PAL["grey"], lw=0.9, ls="--")
ax.annotate("persistence baseline", (1.0, 0), textcoords="offset points",
            xytext=(2, -9), fontsize=7, color=PAL["grey"])
ax.set_xticks(HORIZONS); ax.set_xlabel("Forecast horizon (weeks)")
ax.set_ylabel("MAE reduction vs persistence (%)")
ax.set_xlim(0.9, 4.15); ax.margins(y=0.18)
ax.set_title("Forecast skill by target parameterisation", loc="left")
_save(fig, "fig2_skill_by_horizon")

# --- F3: forecast vs actual with conformal band, two contrasting districts ----
I2 = INTERVALS[(2, "split")]
pick = [MD.groupby("unit")["cases"].mean().idxmax(),
        MD.groupby("unit")["cases"].mean().sort_values().index[len(MD.unit.unique()) // 4]]
fig, axes = plt.subplots(2, 1, figsize=(COL2, 3.6), sharex=True)
for ax, u in zip(axes, pick):
    g = I2[I2.unit == u].sort_values("week_start")
    ax.fill_between(g.week_start, g.lo, g.hi, color=PAL["blue"], alpha=0.20,
                    label="90% conformal interval", linewidth=0)
    ax.plot(g.week_start, g.med, color=PAL["blue"], lw=1.2, label="Forecast (median)")
    ax.plot(g.week_start, g.y, color=PAL["verm"], lw=1.0, ls="--", label="Observed")
    ax.set_ylabel("Cases"); ax.set_title(u, loc="left", fontsize=8.5)
axes[0].legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.02),
               handlelength=1.4, columnspacing=1.6)
axes[-1].set_xlabel("Test seasons 2024–2025")
fig.suptitle("Two-week-ahead forecasts with calibrated intervals", x=0.005, y=1.10,
             ha="left", fontsize=9)
_save(fig, "fig3_forecast_with_intervals")

# --- F4: coverage, marginal AND conditional on district burden -------------
# Dot marks rather than bars: the quantity is a probability read against a
# reference line, not a magnitude from zero, so the axis can show the range that
# matters without the truncation a bar chart would smuggle in. In panel (b) the
# SLOPE is the finding - split conformal falls away with burden just as the
# uncalibrated model does; the group-conditional method stays flat.
METHS = [("raw", PAL["purple"], "o", "Uncalibrated"),
         ("split", PAL["green"], "s", "Split conformal"),
         ("mondrian_adaptive", PAL["blue"], "^", "Group-conditional adaptive")]
TERT_ORDER = ["low burden", "mid burden", "high burden"]
H_COND = 2

fig, (axL, axR) = plt.subplots(1, 2, figsize=(COL2, 2.6), sharey=True)

xs = np.arange(3)
for key, c, mk, lab in METHS:
    v = [calib[(calib.horizon_weeks == h) & (calib.key == key)].empirical_coverage.iloc[0]
         for h in [1, 2, 4]]
    axL.plot(xs, v, color=c, marker=mk, label=lab, lw=1.3)
axL.set_xticks(xs); axL.set_xticklabels([f"{h} wk" for h in [1, 2, 4]])
axL.set_xlim(-0.35, 2.35)
axL.set_xlabel("Forecast horizon", fontsize=8)
axL.set_ylabel("Empirical coverage")
axL.set_title("(a) Marginal", loc="left", fontsize=8.5)

xs = np.arange(len(TERT_ORDER))
for key, c, mk, lab in METHS:
    sub = cond[(cond.horizon_weeks == H_COND) & (cond.method == INT_LABEL[key])]
    v = [sub[sub.burden_tertile == t].coverage.iloc[0] if len(sub[sub.burden_tertile == t]) else np.nan
         for t in TERT_ORDER]
    axR.plot(xs, v, color=c, marker=mk, lw=1.3)
    if np.isfinite(v).all():
        axR.annotate(f"spread {max(v) - min(v):.3f}", (xs[-1], v[-1]), textcoords="offset points",
                     xytext=(7, 0), fontsize=6.8, color=c, va="center", annotation_clip=False)
axR.set_xticks(xs); axR.set_xticklabels(["low", "mid", "high"])
axR.set_xlim(-0.35, 2.95)
axR.set_xlabel("District burden tertile", fontsize=8)
axR.set_title(f"(b) Conditional on burden, h = {H_COND} wk", loc="left", fontsize=8.5)

for ax in (axL, axR):
    ax.axhline(NOMINAL, color=PAL["grey"], lw=1.0, ls="--", zorder=0)
    ax.set_ylim(0.78, 0.935)
axL.annotate("nominal 0.90", (-0.30, NOMINAL), textcoords="offset points", xytext=(0, 4),
             fontsize=7, color=PAL["grey"], ha="left")
axL.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(1.05, -0.22),
           handlelength=1.4, columnspacing=1.8)
_save(fig, "fig4_coverage_calibration")

# --- F5: optimism gap, ROC and PR as separate panels (never a dual axis) -----
m4 = matrix[matrix.condition != "OPTIMISM GAP (C1 - C4)"]
fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.4))
for ax, col, lab in zip(axes, ["ROC_AUC", "PR_AUC"], ["ROC-AUC", "PR-AUC"]):
    cols = [PAL["purple"], PAL["blue"], PAL["orange"], PAL["verm"]]
    ax.bar(range(len(m4)), m4[col], 0.62, color=cols)
    for i, v in enumerate(m4[col]):
        ax.text(i, v + 0.012, f"{v:.3f}", ha="center", fontsize=7)
    ax.set_xticks(range(len(m4)))
    ax.set_xticklabels(["C1\nrandom", "C2\ntime", "C3\nspace", "C4\nboth"])
    ax.set_ylim(0, 1.09); ax.set_ylabel(lab)
axes[0].set_title("Discrimination collapses under honest validation", loc="left")
_save(fig, "fig5_optimism_gap")

# --- F6: lead time is not identifiable ---------------------------------------
# The old version of this figure was a histogram of lead times at one operating
# rule, which presented an artifact as a result. What the data actually supports is
# the spread ACROSS rules, so that is what the figure shows.
if len(LEAD):
    L6 = LEAD.sort_values("median_lead_weeks")
    fig, ax = plt.subplots(figsize=(COL1, 2.5))
    ypos = np.arange(len(L6))
    ax.hlines(ypos, L6.iqr_lo, L6.iqr_hi, color=PAL["green"], lw=3.2, alpha=0.55)
    ax.plot(L6.median_lead_weeks, ypos, "o", color=PAL["verm"], ms=5, zorder=3)
    for i, (_, r) in enumerate(L6.iterrows()):
        ax.text(r.median_lead_weeks, i + 0.24, f"{r.median_lead_weeks:.0f}",
                ha="center", fontsize=7, color=PAL["verm"])
    ax.set_yticks(ypos)
    ax.set_yticklabels([t.replace(" of districts per week", "").replace(" fitted on 2023", "")
                        for t in L6.operating_rule], fontsize=7)
    ax.set_xlabel("Median weeks between first alarm and outbreak onset (bar = IQR)")
    ax.set_xlim(left=0)
    ax.set_title("Lead time is a property of the operating rule, not of the model",
                 loc="left")
    _save(fig, "fig6_alarm_lead_time")

In [ ]:
# =============================================================================
# CELL 15 — Final gate and run manifest
# =============================================================================
HEADLINE_INTERVAL = "mondrian_adaptive"
cov2 = {h: (calib[(calib.horizon_weeks == h) & (calib.key == "raw")].empirical_coverage.iloc[0],
            calib[(calib.horizon_weeks == h) &
                  (calib.key == HEADLINE_INTERVAL)].empirical_coverage.iloc[0])
        for h in [1, 2, 4]}
for h, (raw, conf) in cov2.items():
    # Conformal may legitimately shrink an over-wide interval, so "did it widen?" is the
    # wrong test. The requirement is that it moves coverage TOWARDS nominal.
    assert abs(conf - NOMINAL) <= abs(raw - NOMINAL) + 1e-9, (
        f"h={h}: calibration moved coverage away from nominal "
        f"(raw {raw:.3f} -> conformal {conf:.3f})")
    assert abs(conf - NOMINAL) < 0.06, f"h={h}: coverage {conf:.3f} is far from nominal"
log.info("[gate] calibration OK: %s", {h: round(c, 3) for h, (_, c) in cov2.items()})

# Every constant in this notebook, with how it was fixed. Anything marked
# "assumption" is a judgement call the paper must state rather than bury.
def _floored_at(q):
    """How many districts use the floor rather than their own quantile, from table6b."""
    m = GAPS[(GAPS.alarm_quantile.sub(q).abs() < 1e-9)
             & (GAPS.alarm_floor == ALARM_MIN_CASES)]
    return int(m["districts_at_floor"].iloc[0]) if len(m) else -1


_FLOOR_AT_Q = _floored_at(ALARM_QUANTILE)
_FLOOR_AT_LOW_Q = _floored_at(min(ALARM_Q_SWEEP))

ASSUMPTIONS = pd.DataFrame([
    ("horizons", str(HORIZONS), "operational choice",
     "1-4 weeks is the window in which a health directorate can act"),
    ("test years", str(TEST_YEARS), "empirical",
     f"2026 excluded: {int((MD.year == 2026).sum())} rows over "
     f"{int(MD[MD.year == 2026].epi_week.nunique())} pre-monsoon weeks, not a full season; "
     "used instead as a frozen forward test in Cell 13"),
    ("selection split", f"train<={SEL_TRAIN}, validate {SEL_VALID}", "design",
     "no test year participates in any tuning decision"),
    ("tree hyperparameters", str({k: LGB_REG[k] for k in ('num_leaves', 'min_child_samples')}),
     "SELECTED", "inner-validation MAE sweep, Cell 4b / table1d"),
    ("tweedie_variance_power", f"{TWEEDIE_P}", "SELECTED", "inner-validation MAE sweep over 1.1-1.9"),
    ("growth objective", GROWTH_OBJ_NAME, "SELECTED", "inner-validation MAE, L1 vs L2"),
    ("conformal calibration weeks", str(CAL_WEEKS_BY_H), "SELECTED",
     "inner-validation |coverage - nominal| over 26/39/52, averaged across seeds and chosen "
     "per horizon; floored at 26 because the quantile is estimated from independent weeks"),
    ("adaptive conformal gamma", f"{ACI_GAMMA}", "SELECTED",
     "inner-validation |coverage - nominal| over 0.005-0.10"),
    ("interval method reported", "group-conditional adaptive conformal", "empirical",
     "marginal coverage of a single shared correction was ~0.90 in low-burden districts "
     "and ~0.83 in high-burden ones; calibrating within burden tertiles closes that gap"),
    ("ARIMA order", str(ARIMA_ORDER), "from the literature, OUT OF REGIME",
     "Naher et al. (2022) selected ARIMA(2,1,2) on national MONTHLY counts. Applied "
     "here to district-week counts that are 47% zeros it is outside its design regime "
     "and loses to persistence. Evidence that the published benchmark does not transfer "
     "to operational resolution - NOT evidence that our model beats ARIMA, since the "
     "order was not re-selected for this data"),
    ("paired-comparison seeds", str(N_PAIR_SEEDS), "power analysis",
     "three seeds could not resolve a 2.53-point difference against a 5.95-point "
     "seed sd; the paired design blocks on seed and uses this many"),
    ("seed ensemble", str(SEEDS), "design",
     # Interpolated from table12. A hardcoded "7.6" here was right for one local run and
     # wrong for the canonical one, which spread 11.02 points on the same model.
     f"single-seed estimates span up to "
     f"{(STAB.single_seed_max_pct - STAB.single_seed_min_pct).max():.1f} skill points "
     "on the worst model; every learned "
     "model is the mean over these seeds and table12 reports the member spread"),
    ("nominal coverage", f"{NOMINAL}", "reporting convention", "90% is standard for epidemic forecast hubs"),
    ("significance level", f"{ALPHA}", "convention",
     "0.05 two-sided; every family of tests is Benjamini-Hochberg corrected"),
    ("alarm quantile", f"{ALARM_QUANTILE}", "assumption, SENSITIVITY TESTED",
     f"per-unit 80th percentile of training-year cases; base rate {A[YA].mean():.3f}. "
     "Cell 9b re-measures the optimism gap at 0.70/0.75/0.80/0.85 - see "
     "table6b_alarm_threshold_sensitivity.csv"),
    ("alarm threshold basis", f"[{ALARM_BASE_MIN}, {ALARM_TRAIN_MAX}]", "empirical",
     "the per-district quantile is taken over the modelling window only. Including the "
     "2019 partial season (~30% of that year) shifted every per-district threshold and "
     f"with it the reported optimism gap: this run gives {c1[0] - c4[0]:.4f} ROC against "
     "0.0776 in the last run made on the old basis (committed results/table6, base rate "
     f"0.179 there against {A[YA].mean():.3f} here). A sensitivity sweep that cannot "
     "reproduce the number it perturbs is measuring something else, and the gate in "
     "Cell 9b now checks that it does"),
    ("alarm minimum cases", f"{ALARM_MIN_CASES}", "assumption, SENSITIVITY TESTED",
     # Read from the sweep rather than typed in. A hand-computed version of this said
     # 8 and 25; the modelled frame drops the early-2022 weeks that lack an 8-week lag,
     # and the true counts are lower. Prose numbers drift, interpolated ones cannot.
     "floor so near-zero districts cannot alarm on a single case. It binds for "
     f"{_FLOOR_AT_Q:d} of {MD.unit.nunique()} districts at q={ALARM_QUANTILE} and "
     f"{_FLOOR_AT_LOW_Q:d} at q={min(ALARM_Q_SWEEP)}, so it is not a formality; "
     "table6b sweeps it over 1/3/5/10 and the gap is largest with no floor at all"),
    ("case lags", "[1,2,3,4,8] weeks", "design, a priori",
     "spans the intrinsic (4-7d) plus extrinsic (8-12d) incubation cycle and two "
     "transmission generations. Fixed from the entomology before any result was seen "
     "and never swept, so it cannot be a source of optimism - but it is also not "
     "optimised, and a tuned lag set would likely score higher"),
    ("climate lags", "[2,3,4] weeks", "design, a priori",
     "egg-to-adult development plus the extrinsic incubation period, the standard "
     "2-4 week window in the vector literature. Fixed a priori, same reasoning as the "
     "case lags. table8b leaves each climate variable out in turn; the lag STRUCTURE "
     "itself is not swept and that remains a stated limitation"),
    ("modelling window", "year >= 2022", "empirical",
     "2019 covers only 30% of its season (30,257 of 101,354 official) and 2020-21 are absent"),
    ("reconciliation tolerance", str(RECON_TOL_PCT), "POST-HOC",
     "the divisional bound was widened to 2.5% after observing a -1.85% gap in 2022; "
     "this is a post-hoc choice and is reported as such"),
    ("burden tertiles", "3 equal-count groups", "descriptive", "cut on training-year mean cases only"),
    ("missing-report zeros", f"{len(gap_key)} weeks flagged, retained", "empirical",
     "cases=0 with patients still admitted; Cell 12b shows whether removing them changes the result"),
    ("rainfall source", "district CHIRPS vs divisional NASA POWER", "KNOWN MISMATCH",
     "r=0.86 between products (table1c); Cell 12b re-runs the resolution test without rain"),
    ("bootstrap blocks", "(unit, fold)", "design",
     "resampling whole district-seasons preserves the serial correlation that a row-level "
     "bootstrap would destroy"),
], columns=["constant", "value", "basis", "evidence"])
save_table("table0_assumptions_register", ASSUMPTIONS,
           "Every constant, and whether it was selected empirically or assumed")
print(ASSUMPTIONS.to_string(index=False))

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED, "seed_ensemble": SEEDS, "paired_comparison_seeds": N_PAIR_SEEDS,
    "test_years": TEST_YEARS, "horizons": HORIZONS,
    "alarm": {"quantile": ALARM_QUANTILE, "min_cases": ALARM_MIN_CASES,
              "threshold_window": [ALARM_BASE_MIN, ALARM_TRAIN_MAX]},
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__,
                 "numpy": np.__version__, "lightgbm": lgb.__version__},
    "nominal_coverage": NOMINAL, "calibration_weeks": CAL_WEEKS_BY_H,
    "alpha": ALPHA, "selected_hyperparameters": LGB_REG,
    "selected_tweedie_power": TWEEDIE_P, "selected_growth_objective": GROWTH_OBJ_NAME,
    "selection_split": {"train_max_year": SEL_TRAIN, "validate_year": SEL_VALID},
    "district_rows": int(len(MD)), "district_units": int(MD.unit.nunique()),
    "district_features_full": len(FD_FULL), "district_features_shared": len(FD_SHR),
    "divisional_rows": int(len(MV)) if MV is not None else 0,
    "forbidden_columns_blocked": sorted(FORBIDDEN),
    "inputs": _INPUTS,
    "tables": _TABLES,
    "figures": sorted(os.path.basename(p) for p in glob.glob(os.path.join(FIGURES, "*.png"))),
}
with open(os.path.join(RESULTS, "run_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)

log.info("=" * 78)
log.info("RUN COMPLETE — %d tables, %d figures", len(_TABLES), len(manifest["figures"]))
for k, v in _TABLES.items():
    log.info("  %-34s %3d rows  %s", k, v["rows"], v["caption"])
log.info("=" * 78)